# Afterframe → Wan LoRA: "Sora 2 video, prompt:" fine-tuning on Kaggle 2 × T4

This notebook runs the whole training side of the Afterframe pipeline on Kaggle's free **2 × Tesla T4**:

1. checks the hardware and installs dependencies
2. *(optional)* extracts Afterframe shards inside Kaggle, a few GB at a time
3. downloads the base model (**Wan2.1-T2V-1.3B**; the experimental **Wan2.2-TI2V-5B** preset is optional)
4. caches umT5 text embeddings and VAE latents, with aspect-ratio and frame-count buckets, split across both GPUs
5. runs a **smoke test** end to end (cache, train, resume, export, inference) before the long run
6. benchmarks the chosen preset on these GPUs and estimates the GPU hours the full plan needs
7. trains the LoRA with **DistributedDataParallel** over both GPUs, shard by shard, within a time budget,
   resuming from the last checkpoint in later sessions
8. plots loss, validation loss and GPU memory, and shows validation videos
9. exports the LoRA as `.safetensors` (diffusers layout plus a ComfyUI-keyed copy)
10. **inference**: generates from unseen prompts and compares base vs. LoRA at the same seed

**Precision and attention on T4 (sm_75):** fp16 autocast with a dynamic loss scaler, fp32 LoRA weights, and
no bf16. Attention runs on PyTorch SDPA's memory-efficient and math kernels; FlashAttention is disabled
because it needs sm_80 or newer.

**Two GPUs:** every rank holds a full frozen copy of the base model and its own activations, and only the
LoRA gradients are all-reduced. The pair does not act as one 32 GB device.

**Every training caption is** `Sora 2 video, prompt: <original backend prompt>`. The prefix is included in
the cached text embeddings, and the cache step checks that it survives tokenisation.

Settings: *Accelerator → GPU T4 x2*, *Internet → On*. Optional Kaggle Secrets: `HF_TOKEN` (mirror checkpoints
to a private Hugging Face repo) and `AFTERFRAME_COOKIE` (needed only for extraction inside Kaggle).

## 0 · Configuration

In [ ]:
import os, time
NOTEBOOK_START = time.time()

PRESET = "balanced"          # smoke | low_memory | balanced | max_quality | wan22_5b_experimental
SESSION_HOURS = 11.5         # Kaggle stops GPU sessions at 12 h; leave margin for export and inference
SAFETY_MINUTES = 25          # reserved at the end of the session

DATASET_ROOTS = ["/kaggle/input"]                # searched for shard_*/manifest.jsonl (prepare_dataset.py output)
WORK = "/kaggle/working"                         # the %%writefile cells below write here
OUTPUT_DIR = f"{WORK}/afterframe_run"            # checkpoints, logs, LoRA: saved with the notebook version
CACHE_DIR = "/tmp/afterframe_cache"              # latents/embeddings: large and temporary; NOT under /kaggle/working
SHARDS_PER_CHUNK = 1                             # shards cached + trained together (more = better mixing, more disk)
DELETE_CACHE_AFTER_TRAINING = True               # free disk once a chunk has been trained
STAGE_SHARD_FRACTION = None                      # None = the preset's per-stage shard_fraction; e.g. [0.25, 1.0]
TOTAL_STEPS = None                               # cosine horizon; None = computed from the dataset below

HF_REPO = None                                   # e.g. "your-name/afterframe-lora-ckpts" (private) to mirror checkpoints
RUN_SMOKE_TEST = True
RUN_BENCHMARK = True
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1 · Hardware check

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap,driver_version --format=csv
!df -h /kaggle/working /tmp | sed 's/^/  /'
!free -g | head -2
import torch
NUM_GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "CUDA", torch.version.cuda, "GPUs:", NUM_GPUS)
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {p.name} {p.total_memory/2**30:.1f} GB sm_{p.major}{p.minor}  bf16-native: {(p.major, p.minor) >= (8, 0)}")
assert NUM_GPUS >= 1, "enable a GPU accelerator"
if NUM_GPUS < 2:
    print("WARNING: only one GPU visible; training will run single-process (Accelerator should be 'GPU T4 x2')")

## 2 · Dependencies
Kaggle's preinstalled PyTorch is kept. Only the libraries below are installed or upgraded.

In [ ]:
!pip install -q "diffusers>=0.35.0" "transformers>=4.51.0,<6" "peft>=0.17.0" "accelerate>=1.0" "safetensors>=0.4.3" "huggingface_hub>=0.30" "bitsandbytes>=0.45" ftfy sentencepiece "numpy>=1.26" requests tqdm
!which ffmpeg && ffmpeg -version | head -1
import importlib
for m in ["diffusers", "transformers", "peft", "bitsandbytes", "safetensors"]:
    mod = importlib.import_module(m); print(f"{m:14s}", getattr(mod, "__version__", "?"))

## 3 · Pipeline code
These cells write the same files that are tested in the repository.

In [ ]:
%%writefile /kaggle/working/afterframe_wan.py
#!/usr/bin/env python3
"""
afterframe_wan.py — LoRA fine-tuning of Wan text-to-video models on Afterframe, built for Kaggle's 2 x T4.

Subcommands (all rank-aware; launch with `torchrun --nproc_per_node=2 afterframe_wan.py <cmd> ...`):

  cache    decode clips -> VAE latents, captions -> umT5 embeddings, written per stage and per shard
  train    DDP LoRA training on cached tensors (fp16 autocast + GradScaler, gradient checkpointing,
           gradient accumulation, 8-bit AdamW, resumable, time-budgeted)
  export   turn a checkpoint into a portable .safetensors LoRA (diffusers and ComfyUI key layouts)
  infer    generate videos with / without the LoRA from the same prompt and seed
  presets  print the preset table

Hardware notes (Tesla T4 = Turing, sm_75):
  * no bf16 compute: everything runs fp16 autocast, with LoRA weights and optimizer state in fp32 and a
    dynamic loss scaler; modules diffusers marks _keep_in_fp32_modules stay fp32;
  * no FlashAttention 2: PyTorch SDPA is restricted to its memory-efficient and math kernels, both of
    which support sm_75;
  * the two GPUs are separate 16 GB devices: training is DistributedDataParallel, so every GPU holds its own
    copy of the frozen base model and its own activations; only LoRA gradients are all-reduced.
"""
from __future__ import annotations

import argparse
import contextlib
import datetime as dt
import glob
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import threading
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F

CAPTION_PREFIX = "Sora 2 video, prompt: "
DEFAULT_NEGATIVE = ("Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, "
                    "static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, "
                    "extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, "
                    "fused fingers, still picture, messy background, three legs, many people in the background, "
                    "walking backwards")
DEFAULT_VAL_PROMPTS = [
    CAPTION_PREFIX + "A realistic handheld recording of an abandoned shopping mall at night.",
    CAPTION_PREFIX + "A slow dolly shot through a sunlit greenhouse as mist drifts between hanging ferns.",
    CAPTION_PREFIX + "A street musician plays violin on a rainy Paris bridge while pedestrians hurry past with umbrellas.",
]

# ------------------------------------------------------------------------------------------------
# model registry and presets
# ------------------------------------------------------------------------------------------------
MODELS = {
    "wan21_1.3b": {"repo": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers", "spatial": 16, "fps": 16, "flow_shift": 3.0,
                   "expand_timesteps": False},
    "wan22_5b": {"repo": "Wan-AI/Wan2.2-TI2V-5B-Diffusers", "spatial": 32, "fps": 24, "flow_shift": 5.0,
                 "expand_timesteps": True},
}

TARGETS = {
    # diffusers WanTransformer3DModel module names (regex, full match)
    "self_attn": r"blocks\.\d+\.attn1\.(to_q|to_k|to_v|to_out\.0)",
    "attn": r"blocks\.\d+\.(attn1|attn2)\.(to_q|to_k|to_v|to_out\.0)",
    "attn_ffn": r"blocks\.\d+\.((attn1|attn2)\.(to_q|to_k|to_v|to_out\.0)|ffn\.net\.0\.proj|ffn\.net\.2)",
}


def _buckets(w: int, h: int, sq: int) -> list[list[int]]:
    return [[w, h], [h, w], [sq, sq]]


PRESETS: dict[str, dict] = {
    # tiny end-to-end check on the real model: minutes, not hours
    "smoke": dict(model="wan21_1.3b", rank=8, alpha=8, targets="attn", lr=1e-4, accum=1, batch=1,
                  clips_per_video=1, stages=[dict(name="smoke_192p", buckets=_buckets(320, 192, 256), frames=[17])]),
    # fits comfortably in 16 GB, quickest per sample
    "low_memory": dict(model="wan21_1.3b", rank=16, alpha=16, targets="self_attn", lr=1e-4, accum=4, batch=1,
                       clips_per_video=2,
                       stages=[dict(name="s1_288p_33f", buckets=_buckets(512, 288, 384), frames=[33])]),
    # recommended: progressive 288p -> 352p, attention + cross-attention
    "balanced": dict(model="wan21_1.3b", rank=32, alpha=32, targets="attn", lr=1e-4, accum=4, batch=1,
                     clips_per_video=2,
                     stages=[dict(name="s1_288p_33f", buckets=_buckets(512, 288, 384), frames=[33], shard_fraction=0.5),
                             dict(name="s2_352p_49f", buckets=_buckets(640, 352, 480), frames=[49], shard_fraction=1.0)]),
    # native Wan2.1 480p, 81 frames, attention + FFN; slowest, highest fidelity
    "max_quality": dict(model="wan21_1.3b", rank=64, alpha=64, targets="attn_ffn", lr=5e-5, accum=8, batch=1,
                        clips_per_video=2,
                        stages=[dict(name="s1_288p_33f", buckets=_buckets(512, 288, 384), frames=[33], shard_fraction=0.25),
                                dict(name="s2_352p_49f", buckets=_buckets(640, 352, 480), frames=[49], shard_fraction=1.0),
                                dict(name="s3_480p_81f", buckets=_buckets(832, 480, 624), frames=[81], shard_fraction=0.15)]),
    # Wan2.2 TI2V-5B: stronger base, but 10 GB of fp16 weights per T4. Experimental: smoke-test first.
    "wan22_5b_experimental": dict(model="wan22_5b", rank=32, alpha=32, targets="attn", lr=5e-5, accum=4, batch=1,
                                  clips_per_video=2,
                                  stages=[dict(name="s1_352p_49f", buckets=_buckets(640, 352, 480), frames=[49], shard_fraction=1.0),
                                          dict(name="s2_480p_73f", buckets=_buckets(832, 480, 640), frames=[73], shard_fraction=0.3)]),
}


# ------------------------------------------------------------------------------------------------
# distributed helpers
# ------------------------------------------------------------------------------------------------
@dataclass
class Dist:
    rank: int = 0
    world: int = 1
    local_rank: int = 0
    device: torch.device = field(default_factory=lambda: torch.device("cpu"))

    @property
    def main(self) -> bool:
        return self.rank == 0


def init_dist() -> Dist:
    world = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    local = int(os.environ.get("LOCAL_RANK", "0"))
    cuda = torch.cuda.is_available()
    if cuda:
        torch.cuda.set_device(local)
        device = torch.device("cuda", local)
    else:
        device = torch.device("cpu")
    if world > 1 and not dist.is_initialized():
        # Kaggle's T4 pair talks over PCIe; P2P is disabled by default for robustness (LoRA all-reduce is small).
        os.environ.setdefault("NCCL_P2P_DISABLE", "1")
        dist.init_process_group(backend="nccl" if cuda else "gloo", timeout=dt.timedelta(minutes=90),
                                **({"device_id": device} if cuda else {}))
    return Dist(rank, world, local, device)


def barrier(d: Dist) -> None:
    if d.world > 1:
        dist.barrier()


def all_mean(d: Dist, value: float) -> float:
    if d.world == 1:
        return value
    t = torch.tensor([value], dtype=torch.float64, device=d.device)
    dist.all_reduce(t)
    return float(t.item()) / d.world


def all_sum(d: Dist, values: list[float]) -> list[float]:
    if d.world == 1:
        return values
    t = torch.tensor(values, dtype=torch.float64, device=d.device)
    dist.all_reduce(t)
    return t.tolist()


def gather_obj(d: Dist, obj):
    if d.world == 1:
        return [obj]
    out = [None] * d.world
    dist.all_gather_object(out, obj)
    return out


def log(d: Dist | None, msg: str, all_ranks: bool = False) -> None:
    if d is None or d.main or all_ranks:
        tag = f"[rank{d.rank}]" if d is not None and d.world > 1 else ""
        print(time.strftime("[%H:%M:%S]") + tag + " " + msg, flush=True)


def configure_hardware(d: Dist) -> dict:
    info = {"device": str(d.device), "cuda": torch.cuda.is_available()}
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability(d.device)
        info.update(name=torch.cuda.get_device_name(d.device), capability=f"{cap[0]}.{cap[1]}",
                    total_gb=round(torch.cuda.get_device_properties(d.device).total_memory / 2**30, 2))
        if cap < (8, 0):
            # Turing: FlashAttention kernels need sm_80+. Keep SDPA on mem-efficient (xFormers-style) + math.
            torch.backends.cuda.enable_flash_sdp(False)
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            torch.backends.cuda.enable_math_sdp(True)
            info["attention"] = "sdpa: mem_efficient + math (flash disabled, sm<80)"
        info["bf16_native"] = cap >= (8, 0)
    return info


def amp_ctx(d: Dist, enabled: bool):
    if enabled and d.device.type == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


# ------------------------------------------------------------------------------------------------
# config
# ------------------------------------------------------------------------------------------------
def resolve_config(args) -> dict:
    preset = json.loads(json.dumps(PRESETS[args.preset]))
    for key in ("model", "rank", "alpha", "lr", "accum", "batch", "targets", "clips_per_video"):
        val = getattr(args, key, None)
        if val is not None:
            preset[key] = val
    model = MODELS[preset["model"]]
    preset["model_info"] = model
    preset["model_path"] = getattr(args, "model_path", None) or model["repo"]
    preset["fps"] = getattr(args, "target_fps", None) or model["fps"]
    for st in preset["stages"]:
        for w, h in st["buckets"]:
            assert w % model["spatial"] == 0 and h % model["spatial"] == 0, f"bucket {w}x{h} not /{model['spatial']}"
        for f in st["frames"]:
            assert (f - 1) % 4 == 0, f"frame count {f} must be 4k+1"
    preset["preset"] = args.preset
    return preset


def stage_cfg(cfg: dict, stage: int) -> dict:
    if not 0 <= stage < len(cfg["stages"]):
        raise SystemExit(f"preset {cfg['preset']} has stages 0..{len(cfg['stages']) - 1}")
    return cfg["stages"][stage]


# ------------------------------------------------------------------------------------------------
# dataset discovery (output of prepare_dataset.py, possibly split over several Kaggle input datasets)
# ------------------------------------------------------------------------------------------------
def discover_shards(roots: list[str]) -> dict[str, Path]:
    """Map shard name -> folder holding manifest.jsonl, videos/, captions/."""
    found: dict[str, Path] = {}
    for root in roots:
        for m in sorted(glob.glob(os.path.join(root, "**", "shard_*", "manifest.jsonl"), recursive=True)):
            p = Path(m).parent
            found.setdefault(p.name, p)
    return found


def read_manifest(shard_dir: Path) -> list[dict]:
    rows = []
    with (shard_dir / "manifest.jsonl").open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            if r.get("_type"):
                continue
            caption = (shard_dir / "captions" / (r["id"] + ".txt")).read_text(encoding="utf-8")
            # the caption file is the source of truth for training text; it must equal prefix + backend prompt
            if caption != CAPTION_PREFIX + r["backend_prompt"] or caption != r["training_caption"]:
                raise ValueError(f"caption mismatch for {r['id']} in {shard_dir}")
            r["_caption"] = caption
            r["_video"] = str(shard_dir / "videos" / Path(r["video_path"]).name)
            rows.append(r)
    return rows


# ------------------------------------------------------------------------------------------------
# video decoding, bucketing and clip planning
# ------------------------------------------------------------------------------------------------
def nearest_bucket(w: int, h: int, buckets: list[list[int]]) -> tuple[int, int]:
    ar = math.log(w / h)
    bw, bh = min(buckets, key=lambda b: abs(math.log(b[0] / b[1]) - ar))
    return int(bw), int(bh)


def plan_clips(duration: float, frames_options: list[int], fps: float, max_clips: int) -> list[tuple[float, int]]:
    """Non-overlapping windows (start_seconds, frames). Uses the longest frame bucket that fits, spreads the
    windows evenly, and adds one shorter window from the leftover tail when a smaller bucket fits."""
    opts = sorted(frames_options, reverse=True)
    usable = duration - 0.05
    longest = next((f for f in opts if f / fps <= usable), None)
    if longest is None:
        return []
    span = longest / fps
    n = max(1, min(max_clips, int(usable // span)))
    gap = (usable - n * span) / (n + 1)
    clips = [(gap + i * (span + gap), longest) for i in range(n)]
    left = usable - n * span
    if len(clips) < max_clips:
        tail = next((f for f in opts if f / fps <= left and f < longest), None)
        if tail:
            clips = [(i * span, longest) for i in range(n)] + [(n * span, tail)]
    return clips


def decode_clip(path: str, start: float, frames: int, fps: float, w: int, h: int) -> np.ndarray | None:
    """ffmpeg: seek, resample to `fps`, cover-resize + centre-crop to w x h. Returns uint8 F x H x W x 3."""
    vf = f"fps={fps},scale={w}:{h}:force_original_aspect_ratio=increase:flags=bicubic,crop={w}:{h},setsar=1"
    cmd = ["ffmpeg", "-v", "error", "-nostdin", "-ss", f"{start:.3f}", "-i", path, "-map", "0:v:0",
           "-t", f"{frames / fps + 1.0:.3f}", "-vf", vf, "-frames:v", str(frames), "-pix_fmt", "rgb24",
           "-f", "rawvideo", "-"]
    proc = subprocess.run(cmd, capture_output=True, timeout=600)
    need = frames * w * h * 3
    if proc.returncode != 0 or len(proc.stdout) < need:
        return None
    return np.frombuffer(proc.stdout[:need], np.uint8).reshape(frames, h, w, 3)


def write_mp4(frames: np.ndarray, path: str | Path, fps: float) -> None:
    """frames: F x H x W x 3, uint8 or float in [0, 1]."""
    if frames.dtype != np.uint8:
        frames = (np.clip(frames, 0, 1) * 255).round().astype(np.uint8)
    f, h, w, _ = frames.shape
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    cmd = ["ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{w}x{h}", "-r", str(fps),
           "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "17", "-preset", "medium",
           "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", str(path)]
    subprocess.run(cmd, input=np.ascontiguousarray(frames).tobytes(), check=True)


# ------------------------------------------------------------------------------------------------
# loading model components
# ------------------------------------------------------------------------------------------------
def _from_pretrained(cls, path, dtype, device: torch.device | None = None, **kw):
    """Load straight onto the GPU when possible, so a process never holds a full CPU copy of the weights
    (two ranks x 11 GB of umT5 would not fit in Kaggle's ~30 GB of RAM)."""
    attempts = []
    if device is not None and device.type == "cuda":
        attempts.append({"device_map": {"": str(device)}})
    attempts.append({})
    last = None
    for extra in attempts:
        for dkey in ("torch_dtype", "dtype"):
            try:
                model = cls.from_pretrained(path, **{dkey: dtype}, **extra, **kw)
                return model.to(device) if device is not None else model
            except TypeError as exc:   # older/newer libraries name the dtype argument differently
                last = exc
            except (ValueError, NotImplementedError) as exc:
                last = exc
                break                  # this loader cannot take device_map: try the plain load
    raise last


def ensure_model(cfg: dict, d: Dist, parts: tuple[str, ...]) -> str:
    """Return a local folder for the model, downloading only the requested sub-folders (rank 0 first)."""
    path = cfg["model_path"]
    if os.path.isdir(path):
        return path
    from huggingface_hub import snapshot_download
    patterns = ["model_index.json", "*.txt"] + [f"{p}/*" for p in parts]
    if d.main:
        snapshot_download(path, allow_patterns=patterns)
    barrier(d)
    return snapshot_download(path, allow_patterns=patterns)


def load_text_encoder(path: str, device: torch.device, dtype=torch.float16):
    from transformers import AutoTokenizer, UMT5EncoderModel
    tok = AutoTokenizer.from_pretrained(path, subfolder="tokenizer")
    te = _from_pretrained(UMT5EncoderModel, path, dtype, device, subfolder="text_encoder").eval()
    te.requires_grad_(False)
    return tok, te


def load_vae(path: str, device: torch.device, dtype=torch.float32):
    from diffusers import AutoencoderKLWan
    vae = _from_pretrained(AutoencoderKLWan, path, dtype, device, subfolder="vae").eval()
    vae.requires_grad_(False)
    return vae


def load_transformer(path: str, device: torch.device, dtype=torch.float16):
    from diffusers import WanTransformer3DModel
    tr = _from_pretrained(WanTransformer3DModel, path, dtype, device, subfolder="transformer")
    tr.requires_grad_(False)
    return tr


def latent_stats(vae, device) -> tuple[torch.Tensor, torch.Tensor]:
    z = vae.config.z_dim
    mean = torch.tensor(vae.config.latents_mean, dtype=torch.float32, device=device).view(1, z, 1, 1, 1)
    std = torch.tensor(vae.config.latents_std, dtype=torch.float32, device=device).view(1, z, 1, 1, 1)
    return mean, std


@torch.no_grad()
def encode_prompts(tok, te, prompts: list[str], max_len: int = 512) -> list[dict]:
    """Same text processing as diffusers' WanPipeline._get_t5_prompt_embeds (prompt_clean, pad to max_len,
    attention mask, trim to the real length) so training and inference see identical embeddings.
    Also verifies the caption prefix survives tokenisation intact."""
    try:
        from diffusers.pipelines.wan.pipeline_wan import prompt_clean
    except Exception:  # pragma: no cover
        prompt_clean = lambda s: " ".join(s.split())  # noqa: E731
    device = next(te.parameters()).device
    cleaned = [prompt_clean(p) for p in prompts]
    enc = tok(cleaned, padding="max_length", max_length=max_len, truncation=True, add_special_tokens=True,
              return_attention_mask=True, return_tensors="pt")
    full = tok(cleaned, add_special_tokens=True, truncation=False)["input_ids"]
    prefix_ids = tok(CAPTION_PREFIX.rstrip(), add_special_tokens=False)["input_ids"]
    hidden = te(enc.input_ids.to(device), enc.attention_mask.to(device)).last_hidden_state
    out = []
    for i, p in enumerate(prompts):
        n = int(enc.attention_mask[i].sum())
        emb = hidden[i, :n].float()
        ids = enc.input_ids[i, :n].tolist()
        out.append({
            "embeds": emb.to(torch.float16).cpu(),
            "finite": bool(torch.isfinite(emb).all()),
            "tokens": n, "untruncated_tokens": len(full[i]), "truncated": len(full[i]) > max_len,
            "prefix_tokens_intact": (not p.startswith(CAPTION_PREFIX)) or ids[: len(prefix_ids)] == prefix_ids,
            "cleaned_text": cleaned[i],
        })
    return out


def pad_text(emb: torch.Tensor, max_len: int = 512) -> torch.Tensor:
    if emb.shape[0] >= max_len:
        return emb[:max_len]
    return torch.cat([emb, emb.new_zeros(max_len - emb.shape[0], emb.shape[1])], 0)


# ------------------------------------------------------------------------------------------------
# safetensors I/O
# ------------------------------------------------------------------------------------------------
def save_st(tensors: dict, path: Path, meta: dict | None = None) -> None:
    from safetensors.torch import save_file
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp{os.getpid()}")
    save_file({k: v.contiguous() for k, v in tensors.items()}, str(tmp),
              metadata={k: (v if isinstance(v, str) else json.dumps(v)) for k, v in (meta or {}).items()})
    tmp.replace(path)


def load_st(path: Path) -> tuple[dict, dict]:
    from safetensors import safe_open
    with safe_open(str(path), framework="pt") as fh:
        meta = fh.metadata() or {}
        return {k: fh.get_tensor(k) for k in fh.keys()}, meta


# ------------------------------------------------------------------------------------------------
# CACHE
# ------------------------------------------------------------------------------------------------
def text_key(caption: str) -> str:
    return hashlib.sha256(caption.encode("utf-8")).hexdigest()[:32]


def cmd_cache(args) -> None:
    d = init_dist()
    hw = configure_hardware(d)
    cfg = resolve_config(args)
    st = stage_cfg(cfg, args.stage)
    cache = Path(args.cache_dir)
    shards = discover_shards(args.dataset)
    wanted = sorted(shards) if not args.shards else [s for s in args.shards.split(",") if s]
    missing = [s for s in wanted if s not in shards]
    if missing:
        raise SystemExit(f"shards not found under {args.dataset}: {missing}")
    log(d, f"hardware {hw}; stage {st['name']}; shards {wanted}")
    rows = []
    for s in wanted:
        for r in read_manifest(shards[s]):
            if args.max_videos_per_shard and sum(x["shard"] == s for x in rows) >= args.max_videos_per_shard:
                break
            rows.append(r)
    model_dir = ensure_model(cfg, d, ("tokenizer", "text_encoder", "vae", "scheduler"))
    stats = {"text": 0, "text_s": 0.0, "clips": 0, "clip_s": 0.0, "failed": 0, "truncated": 0}

    # ---- text: one embedding per unique caption, plus negative + validation prompts ------------------
    if "text" in args.what:
        tdir = cache / "text"
        texts = {text_key(r["_caption"]): r["_caption"] for r in rows}
        extras = {"_negative": args.negative_prompt, "_empty": ""}
        for i, p in enumerate(load_val_prompts(args.val_prompts)):
            extras[f"_val{i:02d}"] = p
        todo = [(k, t) for k, t in sorted(texts.items()) + sorted(extras.items()) if not (tdir / f"{k}.safetensors").exists()]
        mine = todo[d.rank::d.world]
        for r in range(d.world):           # ranks load one after another to bound peak CPU RAM
            if r == d.rank and mine:
                tok, te = load_text_encoder(model_dir, d.device)
            barrier(d)
        if mine:
            t0 = time.time()
            for i in range(0, len(mine), args.text_batch):
                chunk = mine[i:i + args.text_batch]
                res = encode_prompts(tok, te, [t for _, t in chunk])
                for (k, t), r in zip(chunk, res):
                    if not r["finite"]:
                        raise SystemExit("umT5 produced non-finite fp16 embeddings; rerun cache with --text-fp32-cpu")
                    if not r["prefix_tokens_intact"]:
                        raise SystemExit(f"caption prefix was altered by tokenisation for: {t[:80]!r}")
                    stats["truncated"] += r["truncated"]
                    save_st({"embeds": r["embeds"]}, tdir / f"{k}.safetensors",
                            {"text": t, "tokens": str(r["tokens"]), "untruncated_tokens": str(r["untruncated_tokens"]),
                             "truncated": str(r["truncated"])})
                stats["text"] += len(chunk)
                if (i // args.text_batch) % 20 == 0:
                    log(d, f"text {stats['text']}/{len(mine)} on this rank", all_ranks=True)
            stats["text_s"] = time.time() - t0
            del te
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
        barrier(d)

    # ---- latents -------------------------------------------------------------------------------------
    if "latents" in args.what:
        vae = load_vae(model_dir, d.device, torch.float32)
        if hasattr(vae, "enable_tiling") and args.vae_tiling:
            vae.enable_tiling()
        mean, std = latent_stats(vae, d.device)
        vae16 = None
        vae_dtype = {"fp32": torch.float32, "fp16": torch.float16}.get(args.vae_dtype)
        jobs = []
        for r in rows:
            bw, bh = nearest_bucket(r["width"], r["height"], st["buckets"])
            for ci, (start, nf) in enumerate(plan_clips(r["duration"], st["frames"], cfg["fps"], cfg["clips_per_video"])):
                key = f"{r['id']}_c{ci}_{bw}x{bh}x{nf}"
                # val clips live in one shared folder so deleting a trained shard's cache keeps them
                group = "val" if r["split"] == "val" else r["shard"]
                out = cache / st["name"] / group / "latents" / f"{key}.safetensors"
                jobs.append((r, key, start, nf, bw, bh, out))
        todo = [j for j in jobs if not j[-1].exists()]
        mine = todo[d.rank::d.world]
        log(d, f"{len(jobs)} clips planned, {len(todo)} to encode ({len(mine)} on rank {d.rank})", all_ranks=True)

        def encode(x, dtype):
            model = vae if dtype == torch.float32 else vae16
            z = model.encode(x.to(dtype)).latent_dist.mode().float()
            return (z - mean) / std

        # prefetch decoding on a CPU thread while the GPU encodes
        from concurrent.futures import ThreadPoolExecutor
        pool = ThreadPoolExecutor(max_workers=args.decode_workers)
        futs = [pool.submit(decode_clip, j[0]["_video"], j[2], j[3], cfg["fps"], j[4], j[5]) for j in mine[:args.decode_workers * 2]]
        nxt = len(futs)
        t0 = time.time()
        for idx, job in enumerate(mine):
            r, key, start, nf, bw, bh, out = job
            frames = futs[idx].result()
            futs[idx] = None
            if nxt < len(mine):
                j = mine[nxt]
                futs.append(pool.submit(decode_clip, j[0]["_video"], j[2], j[3], cfg["fps"], j[4], j[5]))
                nxt += 1
            if frames is None:
                stats["failed"] += 1
                log(d, f"decode failed: {key}", all_ranks=True)
                continue
            x = torch.from_numpy(frames).to(d.device).permute(3, 0, 1, 2).unsqueeze(0).float().div_(127.5).sub_(1)
            with torch.no_grad():
                if vae_dtype is None:   # "auto": test fp16 against fp32 once, then commit
                    ref = encode(x, torch.float32)
                    vae16 = load_vae(model_dir, d.device, torch.float16)
                    if hasattr(vae16, "enable_tiling") and args.vae_tiling:
                        vae16.enable_tiling()
                    t16 = encode(x, torch.float16)
                    err = float((t16 - ref).abs().mean() / ref.abs().mean())
                    ok = bool(torch.isfinite(t16).all()) and err < args.vae_fp16_tolerance
                    vae_dtype = torch.float16 if ok else torch.float32
                    log(d, f"VAE fp16 vs fp32 relative error {err:.4f} -> using {'fp16' if ok else 'fp32'}", all_ranks=True)
                    if not ok:
                        del vae16
                        vae16 = None
                    z = ref
                else:
                    if vae_dtype == torch.float16 and vae16 is None:
                        vae16 = load_vae(model_dir, d.device, torch.float16)
                        if hasattr(vae16, "enable_tiling") and args.vae_tiling:
                            vae16.enable_tiling()
                    z = encode(x, vae_dtype)
            if not torch.isfinite(z).all():
                stats["failed"] += 1
                log(d, f"non-finite latents: {key}", all_ranks=True)
                continue
            save_st({"latents": z[0].to(torch.float16).cpu()}, out,
                    {"post_id": r["id"], "shard": r["shard"], "split": r["split"], "text_key": text_key(r["_caption"]),
                     "start": f"{start:.3f}", "frames": str(nf), "width": str(bw), "height": str(bh),
                     "fps": str(cfg["fps"]), "model": cfg["model"], "dup_group": r["dup_group"]})
            stats["clips"] += 1
            if idx % 25 == 0:
                el = time.time() - t0
                log(d, f"latents {idx + 1}/{len(mine)}  {el / (idx + 1):.2f}s/clip", all_ranks=True)
        pool.shutdown()
        stats["clip_s"] = time.time() - t0
        barrier(d)
        if d.main:
            write_index(cache / st["name"], wanted + ["val"])

    allstats = gather_obj(d, stats)
    if d.main:
        bench = {"kind": "cache", "time": time.time(), "preset": cfg["preset"], "stage": st["name"], "hardware": hw,
                 "world": d.world, "per_rank": allstats,
                 "sec_per_clip": [s["clip_s"] / s["clips"] for s in allstats if s["clips"]],
                 "sec_per_text": [s["text_s"] / s["text"] for s in allstats if s["text"]]}
        with (cache / "benchmarks.jsonl").open("a") as fh:
            fh.write(json.dumps(bench) + "\n")
        log(d, f"cache done: {bench['per_rank']}")
    barrier(d)


def write_index(stage_dir: Path, shards: list[str]) -> None:
    """index.jsonl per shard lists every cached clip (read from safetensors headers)."""
    from safetensors import safe_open
    for s in shards:
        if not (stage_dir / s / "latents").exists():
            continue
        rows = []
        for p in sorted((stage_dir / s / "latents").glob("*.safetensors")):
            with safe_open(str(p), framework="pt") as fh:
                meta = fh.metadata()
                shape = list(fh.get_slice("latents").get_shape())
            rows.append({"key": p.stem, "path": str(p.relative_to(stage_dir)), "shape": shape, **meta})
        (stage_dir / s).mkdir(parents=True, exist_ok=True)
        with (stage_dir / s / "index.jsonl").open("w") as fh:
            for r in rows:
                fh.write(json.dumps(r) + "\n")


def load_val_prompts(path: str | None) -> list[str]:
    if not path:
        return list(DEFAULT_VAL_PROMPTS)
    return [ln.rstrip("\n") for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


# ------------------------------------------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------------------------------------------
class ClipSet(torch.utils.data.Dataset):
    def __init__(self, entries: list[dict], stage_dir: Path, text_dir: Path):
        self.entries, self.stage_dir, self.text_dir = entries, stage_dir, text_dir

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, i):
        e = self.entries[i]
        lat, _ = load_st(self.stage_dir / e["path"])
        txt, _ = load_st(self.text_dir / f"{e['text_key']}.safetensors")
        return {"latents": lat["latents"], "text": pad_text(txt["embeds"]), "key": e["key"]}


def collate(items):
    return {"latents": torch.stack([i["latents"] for i in items]), "text": torch.stack([i["text"] for i in items]),
            "keys": [i["key"] for i in items]}


def load_entries(stage_dir: Path, shards: list[str], split: str, required: bool = True) -> list[dict]:
    out = []
    for s in shards:
        idx = stage_dir / s / "index.jsonl"
        if not idx.exists():
            if not required:
                continue
            raise SystemExit(f"no cache index for {s} in {stage_dir}; run `cache` first")
        for line in idx.read_text().splitlines():
            e = json.loads(line)
            if e["split"] == split:
                out.append(e)
    return out


def make_batches(entries: list[dict], batch: int, seed: int) -> list[list[int]]:
    """Group by bucket (identical tensor shapes), chunk into batches, shuffle batches."""
    rng = random.Random(seed)
    groups: dict[str, list[int]] = {}
    for i, e in enumerate(entries):
        groups.setdefault(json.dumps(e["shape"]), []).append(i)
    batches = []
    for idxs in groups.values():
        rng.shuffle(idxs)
        batches += [idxs[j:j + batch] for j in range(0, len(idxs) - batch + 1, batch)]
    rng.shuffle(batches)
    return batches


def lr_lambda_factory(warmup: int, total: int, min_ratio: float, kind: str):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        if kind == "constant":
            return 1.0
        prog = min(1.0, (step - warmup) / max(1, total - warmup))
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * prog))
    return f


def sample_sigmas(n: int, shift: float, mode: str, gen: torch.Generator, device) -> torch.Tensor:
    if mode == "logit_normal":
        u = torch.sigmoid(torch.randn(n, generator=gen))
    else:
        u = torch.rand(n, generator=gen)
    u = u.clamp(1e-3, 1 - 1e-3)
    return (shift * u / (1 + (shift - 1) * u)).to(device)


def flow_inputs(x0: torch.Tensor, sigma: torch.Tensor, noise: torch.Tensor, expand: bool):
    s = sigma.view(-1, 1, 1, 1, 1)
    xt = (1 - s) * x0 + s * noise
    target = noise - x0
    t = sigma * 1000.0
    if expand:   # Wan2.2 TI2V: one timestep per latent token (all equal for text-to-video)
        b, _, f, h, w = x0.shape
        t = t.view(b, 1).expand(b, f * (h // 2) * (w // 2))
    return xt, target, t


def find_checkpoints(roots: list[str]) -> list[Path]:
    found = []
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "step_*", "training_state.pt"), recursive=True):
            found.append(Path(p).parent)
    return sorted(set(found), key=lambda p: int(re.search(r"step_(\d+)", p.name).group(1)))


def lora_state(model) -> dict[str, torch.Tensor]:
    from peft.utils import get_peft_model_state_dict
    return {k: v.detach().float().cpu() for k, v in get_peft_model_state_dict(model).items()}


def save_checkpoint(out_dir: Path, step: int, model, optimizer, scaler, sched, state: dict, cfg: dict,
                    keep: int, d: Dist, hub=None) -> Path:
    ckdir = out_dir / f"step_{step:07d}"
    if d.main:
        tmp = out_dir / f".tmp_step_{step:07d}"
        shutil.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        save_st(lora_state(model), tmp / "lora_peft.safetensors",
                {"rank": str(cfg["rank"]), "alpha": str(cfg["alpha"]), "targets": cfg["targets"], "step": str(step)})
        torch.save({"optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(), "sched": sched.state_dict(),
                    "state": state, "rng": {"python": random.getstate(), "torch": torch.get_rng_state(),
                                            "numpy": np.random.get_state()}}, tmp / "training_state.pt")
        (tmp / "config.json").write_text(json.dumps(cfg, indent=1))
        shutil.rmtree(ckdir, ignore_errors=True)
        tmp.rename(ckdir)
        (out_dir / "latest.txt").write_text(ckdir.name)
        for old in find_checkpoints([str(out_dir)])[:-keep]:
            shutil.rmtree(old, ignore_errors=True)
        if hub:
            hub(ckdir)
    barrier(d)
    return ckdir


def make_hub_uploader(repo: str | None):
    if not repo:
        return None
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(repo, private=True, exist_ok=True)
    lock = threading.Lock()

    def upload(ckdir: Path):
        def run():
            with lock:
                try:
                    api.upload_folder(repo_id=repo, folder_path=str(ckdir), path_in_repo=f"checkpoints/{ckdir.name}")
                    print(f"[hub] uploaded {ckdir.name} to {repo}", flush=True)
                except Exception as exc:  # never kill training over an upload
                    print(f"[hub] upload failed: {exc}", flush=True)
        threading.Thread(target=run, daemon=True).start()
    return upload


def build_optimizer(params, name: str, lr: float, wd: float):
    if name == "adamw8bit":
        try:
            import bitsandbytes as bnb
            return bnb.optim.AdamW8bit(params, lr=lr, betas=(0.9, 0.999), weight_decay=wd), "adamw8bit"
        except Exception as exc:
            print(f"bitsandbytes unavailable ({exc}); falling back to torch AdamW", flush=True)
    return torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.999), weight_decay=wd), "adamw"


def cmd_train(args) -> None:
    d = init_dist()
    hw = configure_hardware(d)
    if args.mixed_precision == "bf16":
        raise SystemExit("bf16 is not supported on T4 (sm_75); use fp16")
    use_amp = args.mixed_precision == "fp16" and d.device.type == "cuda"
    cfg = resolve_config(args)
    st = stage_cfg(cfg, args.stage)
    cache, out = Path(args.cache_dir), Path(args.output_dir)
    stage_dir, text_dir = cache / st["name"], cache / "text"
    shards = sorted(s for s in args.shards.split(",") if s) if args.shards else \
        sorted(p.name for p in stage_dir.glob("shard_*") if (p / "index.jsonl").exists())
    uncached = [s for s in shards if not (stage_dir / s / "index.jsonl").exists()]
    if uncached:
        raise SystemExit(f"shards {uncached} are not cached for stage {st['name']} in {stage_dir}")
    if not shards:
        raise SystemExit(f"no cached shards in {stage_dir}")
    random.seed(args.seed + d.rank)
    torch.manual_seed(args.seed + d.rank)
    np.random.seed(args.seed + d.rank)

    # ---- model + LoRA ----------------------------------------------------------------------------
    from peft import LoraConfig
    from peft.utils import set_peft_model_state_dict
    model_dir = ensure_model(cfg, d, ("transformer", "vae", "scheduler") if args.val_every else ("transformer",))
    for r in range(d.world):
        if r == d.rank:
            transformer = load_transformer(model_dir, d.device, torch.float16 if d.device.type == "cuda" else torch.float32)
        barrier(d)
    if args.gradient_checkpointing:
        transformer.enable_gradient_checkpointing()
    lcfg = LoraConfig(r=cfg["rank"], lora_alpha=cfg["alpha"], lora_dropout=args.lora_dropout,
                      init_lora_weights=True, target_modules=TARGETS[cfg["targets"]])
    transformer.add_adapter(lcfg)
    trainable = [p for n, p in transformer.named_parameters() if "lora_" in n]
    for p in trainable:
        p.data = p.data.float()       # fp32 master weights for the adapter (required by GradScaler)
        p.requires_grad_(True)
    n_train = sum(p.numel() for p in trainable)
    log(d, f"hardware {hw}")
    log(d, f"LoRA rank {cfg['rank']} alpha {cfg['alpha']} on '{cfg['targets']}': {n_train / 1e6:.2f}M trainable params")

    optimizer, opt_name = build_optimizer(trainable, args.optimizer, cfg["lr"], args.weight_decay)
    total = args.total_steps
    sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda_factory(args.warmup_steps, total, args.min_lr_ratio,
                                                                           args.lr_scheduler))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp, init_scale=2.0 ** 14)

    # ---- resume ------------------------------------------------------------------------------------
    state = {"global_step": 0, "stage_steps": {}, "chunk": None, "pass": 0, "pos": 0, "completed": [],
             "samples": 0, "skipped_nonfinite": 0}
    roots = [str(out / "checkpoints")] + (args.resume_from or [])
    cks = find_checkpoints(roots) if args.resume else []
    if cks:
        ck = cks[-1]
        lora, _ = load_st(ck / "lora_peft.safetensors")
        res = set_peft_model_state_dict(transformer, {k: v.to(d.device) for k, v in lora.items()})
        unexpected = getattr(res, "unexpected_keys", [])
        if unexpected:
            raise SystemExit(f"checkpoint does not match this LoRA config: {unexpected[:3]}")
        for p in trainable:
            p.data = p.data.float()
        blob = torch.load(ck / "training_state.pt", map_location="cpu", weights_only=False)
        optimizer.load_state_dict(blob["optimizer"])
        scaler.load_state_dict(blob["scaler"])
        sched.load_state_dict(blob["sched"])
        state.update(blob["state"])
        log(d, f"resumed from {ck} at global step {state['global_step']}")
    else:
        log(d, "starting from the base model (no checkpoint found)" if args.resume else "fresh run (--no-resume)")

    model = transformer
    if d.world > 1:
        model = torch.nn.parallel.DistributedDataParallel(transformer, device_ids=[d.local_rank] if d.device.type == "cuda" else None,
                                                          gradient_as_bucket_view=True, find_unused_parameters=False)

    # ---- data ------------------------------------------------------------------------------------------
    chunk = ",".join(shards)
    if state["chunk"] != chunk:
        if state["chunk"] and state["pos"]:
            log(d, f"note: previous chunk {state['chunk']} stopped at batch {state['pos']}; starting chunk {chunk}")
        state.update(chunk=chunk, pos=0)
        state["pass"] = 0
    train_entries = load_entries(stage_dir, shards, "train")
    if args.max_clips:
        train_entries = train_entries[: args.max_clips]
    # fixed validation subset: ordered by a hash of the clip key, so it only grows as shards are cached
    val_entries = sorted(load_entries(stage_dir, ["val"], "val", required=False), key=lambda e: hash_str(e["key"]))
    val_entries = val_entries[: args.val_clips]
    trainset, valset = ClipSet(train_entries, stage_dir, text_dir), ClipSet(val_entries, stage_dir, text_dir)
    log(d, f"stage {st['name']}: {len(train_entries)} train clips in {shards}; {len(val_entries)} val clips")
    if not train_entries:
        raise SystemExit("no training clips")

    expand = cfg["model_info"]["expand_timesteps"]
    shift = args.timestep_shift or cfg["model_info"]["flow_shift"]
    log_path = out / "logs" / "train_log.jsonl"
    if d.main:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        (out / "run_config.json").write_text(json.dumps({"cfg": cfg, "args": vars(args), "hardware": hw,
                                                        "optimizer": opt_name, "world": d.world}, indent=1, default=str))
    hub = make_hub_uploader(args.hub_repo) if d.main else None
    t_start = time.time()
    deadline = t_start + args.max_train_minutes * 60 if args.max_train_minutes else None
    stop_reason = None
    accum = cfg["accum"]
    micro_bs = cfg["batch"]
    gen_noise = torch.Generator().manual_seed(args.seed * 1000 + d.rank + state["global_step"])
    step_t0, window_samples = time.time(), 0
    validator = Validator(args, cfg, st, model_dir, d, text_dir, valset, expand, shift) if args.val_every or args.val_loss_every else None

    while stop_reason is None:
        batches = make_batches(train_entries, micro_bs, args.seed + 7919 * state["pass"] + hash_str(chunk))
        per_rank = len(batches) // d.world
        n_opt = per_rank // accum
        usable = n_opt * accum * d.world              # identical on every rank
        if n_opt == 0:
            raise SystemExit(f"too few clips ({len(batches)} batches) for world {d.world} x accum {accum}")
        loader_order = batches[:usable]
        pos = state["pos"]
        # each optimizer step consumes world*accum batches; rank r takes every world-th batch
        mine = [loader_order[i] for i in range(pos + d.rank, usable, d.world)]
        dl = torch.utils.data.DataLoader(torch.utils.data.Subset(trainset, [i for b in mine for i in b]),
                                         batch_size=micro_bs, shuffle=False, num_workers=args.num_workers,
                                         collate_fn=collate, pin_memory=d.device.type == "cuda",
                                         persistent_workers=False, prefetch_factor=2 if args.num_workers else None)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        micro = 0
        loss_acc, n_acc = 0.0, 0
        for batch in dl:
            micro += 1
            last = micro % accum == 0
            x0 = batch["latents"].to(d.device, non_blocking=True).float()
            text = batch["text"].to(d.device, non_blocking=True)
            noise = torch.randn(x0.shape, generator=gen_noise).to(d.device)
            sigma = sample_sigmas(x0.shape[0], shift, args.timestep_sampling, gen_noise, d.device)
            xt, target, t = flow_inputs(x0, sigma, noise, expand)
            sync = contextlib.nullcontext() if (last or d.world == 1) else model.no_sync()
            with sync:
                with amp_ctx(d, use_amp):
                    pred = model(hidden_states=xt.to(torch.float16 if use_amp else torch.float32),
                                 timestep=t, encoder_hidden_states=text.to(torch.float16 if use_amp else torch.float32),
                                 return_dict=False)[0]
                loss = F.mse_loss(pred.float(), target.float())
                # a non-finite loss is still back-propagated: the all-reduced gradients become non-finite on
                # every rank, and the step below is skipped everywhere (GradScaler also lowers its scale)
                scaler.scale(loss / accum).backward()
            loss_acc += float(loss.detach())
            n_acc += 1
            window_samples += x0.shape[0] * d.world
            if not last:
                continue
            scaler.unscale_(optimizer)
            gnorm = float(torch.nn.utils.clip_grad_norm_(trainable, args.max_grad_norm))
            if math.isfinite(gnorm):
                scaler.step(optimizer)
                sched.step()
            else:
                state["skipped_nonfinite"] += 1
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            state["global_step"] += 1
            state["stage_steps"][st["name"]] = state["stage_steps"].get(st["name"], 0) + 1
            state["pos"] += accum * d.world
            state["samples"] += accum * d.world * micro_bs
            gs = state["global_step"]

            if gs % args.log_every == 0 or gs == 1:
                mean_loss = all_mean(d, loss_acc / max(1, n_acc))
                mem = gather_obj(d, round(torch.cuda.max_memory_allocated(d.device) / 2**30, 2)
                                 if d.device.type == "cuda" else 0.0)
                el = time.time() - step_t0
                rec = {"step": gs, "stage": st["name"], "loss": mean_loss, "lr": sched.get_last_lr()[0],
                       "grad_norm": gnorm, "loss_scale": scaler.get_scale() if use_amp else 1.0,
                       "samples_per_s": window_samples / el if el else None,
                       "sec_per_opt_step": el / max(1, (gs % args.log_every) or args.log_every) if gs > 1 else el,
                       "peak_mem_gb": mem, "elapsed_min": (time.time() - t_start) / 60,
                       "pass": state["pass"], "pos": state["pos"], "of": usable, "nonfinite": state["skipped_nonfinite"]}
                if d.main:
                    with log_path.open("a") as fh:
                        fh.write(json.dumps(rec) + "\n")
                    log(d, f"step {gs} loss {mean_loss:.4f} lr {rec['lr']:.2e} gnorm {gnorm:.3f} "
                           f"{rec['samples_per_s'] or 0:.2f} samples/s mem {mem} GB  [{state['pos']}/{usable}]")
                step_t0, window_samples = time.time(), 0
            loss_acc, n_acc = 0.0, 0

            if validator and args.val_loss_every and gs % args.val_loss_every == 0:
                validator.val_loss(model, gs, out)
            if validator and args.val_every and gs % args.val_every == 0:
                validator.generate(transformer, gs, out)
            if gs % args.save_every == 0:
                save_checkpoint(out / "checkpoints", gs, transformer, optimizer, scaler, sched, state, cfg,
                                args.keep_checkpoints, d, hub)
            if args.max_steps and gs >= args.max_steps:
                stop_reason = "max_steps"
            elif deadline and time.time() > deadline:
                stop_reason = "time_budget"
            if stop_reason:
                break
        if stop_reason:
            break
        # finished a pass over the chunk
        state["pass"] += 1
        state["pos"] = 0
        if state["pass"] >= args.passes:
            state["completed"].append({"chunk": chunk, "stage": st["name"], "passes": state["pass"],
                                       "at_step": state["global_step"]})
            stop_reason = "chunk_done"

    log(d, f"stopping: {stop_reason} at step {state['global_step']}")
    sums = gather_obj(d, float(sum(p.detach().double().sum() for p in trainable)))
    in_sync = max(sums) - min(sums) <= 1e-6 * max(1.0, abs(sums[0]))
    log(d, f"LoRA checksum per rank {sums}: {'ranks in sync' if in_sync else 'WARNING: ranks diverged'}")
    if validator and args.val_loss_every and state["global_step"] % args.val_loss_every:
        validator.val_loss(model, state["global_step"], out)
    ck = save_checkpoint(out / "checkpoints", state["global_step"], transformer, optimizer, scaler, sched, state, cfg,
                         args.keep_checkpoints, d, hub)
    if d.main:
        export_lora(ck, out / "afterframe_sora2_lora.safetensors", cfg)
        (out / "last_stop.json").write_text(json.dumps({"reason": stop_reason, "step": state["global_step"],
                                                       "checkpoint": str(ck), "state": state}, indent=1))
    if validator and args.val_every and stop_reason != "time_budget":
        validator.generate(transformer, state["global_step"], out)
    barrier(d)
    if d.world > 1:
        dist.destroy_process_group()


def hash_str(s: str) -> int:
    return int(hashlib.sha1(s.encode()).hexdigest()[:8], 16)


class Validator:
    """Held-out flow-matching loss (fixed noise and timesteps) and sample videos from unseen prompts."""

    SIGMAS = (0.1, 0.3, 0.5, 0.7, 0.9)

    def __init__(self, args, cfg, st, model_dir, d, text_dir, valset, expand, shift):
        self.args, self.cfg, self.st, self.model_dir, self.d = args, cfg, st, model_dir, d
        self.text_dir, self.valset, self.expand, self.shift = text_dir, valset, expand, shift
        self.vae = None

    @torch.no_grad()
    def val_loss(self, model, step: int, out: Path) -> None:
        d = self.d
        if not len(self.valset):
            return
        model.eval()
        tot, cnt = 0.0, 0
        for i in range(d.rank, len(self.valset), d.world):
            item = self.valset[i]
            x0 = item["latents"].unsqueeze(0).to(d.device).float()
            text = item["text"].unsqueeze(0).to(d.device)
            g = torch.Generator().manual_seed(1234 + i)
            noise = torch.randn(x0.shape, generator=g).to(d.device)
            for s in self.SIGMAS:
                sigma = torch.tensor([s], device=d.device)
                xt, target, t = flow_inputs(x0, sigma, noise, self.expand)
                with amp_ctx(d, d.device.type == "cuda"):
                    inner = model.module if hasattr(model, "module") else model
                    dt_ = torch.float16 if d.device.type == "cuda" else torch.float32
                    pred = inner(hidden_states=xt.to(dt_), timestep=t, encoder_hidden_states=text.to(dt_),
                                 return_dict=False)[0]
                tot += float(F.mse_loss(pred.float(), target))
                cnt += 1
        tot, cnt = all_sum(d, [tot, cnt])
        model.train()
        if d.main and cnt:
            with (out / "logs" / "val_log.jsonl").open("a") as fh:
                fh.write(json.dumps({"step": step, "val_loss": tot / cnt, "clips": len(self.valset),
                                     "sigmas": self.SIGMAS}) + "\n")
            log(d, f"validation loss @ {step}: {tot / cnt:.4f}")

    @torch.no_grad()
    def generate(self, transformer, step: int, out: Path) -> None:
        d = self.d
        if d.main:
            try:
                from diffusers import UniPCMultistepScheduler, WanPipeline
                if self.vae is None:
                    self.vae = load_vae(self.model_dir, d.device, torch.float32)
                    if hasattr(self.vae, "enable_tiling"):
                        self.vae.enable_tiling()
                sched = UniPCMultistepScheduler.from_pretrained(self.model_dir, subfolder="scheduler")
                sched = UniPCMultistepScheduler.from_config(sched.config, flow_shift=self.shift)
                pipe = WanPipeline(tokenizer=None, text_encoder=None, vae=self.vae, scheduler=sched,
                                   transformer=transformer, expand_timesteps=self.expand)
                pipe.set_progress_bar_config(disable=True)
                neg, _ = load_st(self.text_dir / "_negative.safetensors")
                transformer.eval()
                w, h = self.st["buckets"][0]
                frames = self.args.val_frames or self.st["frames"][0]
                for k in range(len(load_val_prompts(self.args.val_prompts))):
                    path = self.text_dir / f"_val{k:02d}.safetensors"
                    if not path.exists():
                        continue
                    emb, meta = load_st(path)
                    dt_ = torch.float16 if d.device.type == "cuda" else torch.float32
                    with amp_ctx(d, d.device.type == "cuda"):
                        video = pipe(prompt_embeds=pad_text(emb["embeds"]).unsqueeze(0).to(d.device, dt_),
                                     negative_prompt_embeds=pad_text(neg["embeds"]).unsqueeze(0).to(d.device, dt_),
                                     height=h, width=w, num_frames=frames, num_inference_steps=self.args.val_steps,
                                     guidance_scale=self.args.val_guidance,
                                     generator=torch.Generator(d.device).manual_seed(42 + k), output_type="np").frames[0]
                    dst = out / "validation" / f"step_{step:07d}" / f"prompt{k:02d}.mp4"
                    write_mp4(video, dst, self.cfg["fps"])
                    (dst.with_suffix(".txt")).write_text(meta.get("text", ""), encoding="utf-8")
                    log(d, f"validation video -> {dst}")
                del pipe
            except Exception as exc:   # a failed preview must not end a long run
                log(d, f"validation generation failed: {type(exc).__name__}: {exc}")
            finally:
                transformer.train()
                if d.device.type == "cuda":
                    torch.cuda.empty_cache()
        barrier(d)


# ------------------------------------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------------------------------------
_COMFY_MAP = [(r"\.attn1\.to_q\.", ".self_attn.q."), (r"\.attn1\.to_k\.", ".self_attn.k."),
              (r"\.attn1\.to_v\.", ".self_attn.v."), (r"\.attn1\.to_out\.0\.", ".self_attn.o."),
              (r"\.attn2\.to_q\.", ".cross_attn.q."), (r"\.attn2\.to_k\.", ".cross_attn.k."),
              (r"\.attn2\.to_v\.", ".cross_attn.v."), (r"\.attn2\.to_out\.0\.", ".cross_attn.o."),
              (r"\.ffn\.net\.0\.proj\.", ".ffn.0."), (r"\.ffn\.net\.2\.", ".ffn.2.")]


def export_lora(ckpt: Path, dst: Path, cfg: dict | None = None, dtype=torch.float16) -> dict:
    """Write portable LoRA files. The alpha/rank scale is folded into lora_B, so any loader that applies
    the LoRA at scale 1.0 (diffusers' default when no alpha is stored) reproduces training exactly."""
    lora, meta = load_st(ckpt / "lora_peft.safetensors")
    rank, alpha = int(meta["rank"]), float(meta["alpha"])
    scale = alpha / rank
    diff = {}
    for k, v in lora.items():
        v = v.float() * (scale if ".lora_B." in k else 1.0)
        diff["transformer." + k] = v.to(dtype)
    info = {"format": "diffusers-peft", "base_model": (cfg or {}).get("model_path", ""), "rank": str(rank),
            "alpha_folded": str(alpha), "trigger": CAPTION_PREFIX.strip(), "step": meta.get("step", ""),
            "targets": meta.get("targets", ""), "note": "alpha/rank folded into lora_B; load at strength 1.0"}
    save_st(diff, dst, info)
    comfy = {}
    for k, v in diff.items():
        ck = "diffusion_model." + k[len("transformer."):]
        for pat, rep in _COMFY_MAP:
            ck = re.sub(pat, rep, ck)
        comfy[ck] = v
    save_st(comfy, dst.with_name(dst.stem + "_comfyui.safetensors"), {**info, "format": "comfyui-wan"})
    return info


def cmd_export(args) -> None:
    roots = args.checkpoint_roots or ["outputs/checkpoints"]
    cks = [Path(args.checkpoint)] if args.checkpoint else find_checkpoints(roots)
    if not cks:
        raise SystemExit("no checkpoint found")
    cfg = json.loads((cks[-1] / "config.json").read_text()) if (cks[-1] / "config.json").exists() else None
    info = export_lora(cks[-1], Path(args.out), cfg)
    print(json.dumps({"checkpoint": str(cks[-1]), "out": args.out, **info}, indent=1))


# ------------------------------------------------------------------------------------------------
# INFERENCE
# ------------------------------------------------------------------------------------------------
class Generator:
    """Transformer + VAE on one GPU, umT5 on the other (fp16 umT5 alone is ~11 GB)."""

    def __init__(self, model: str = "wan21_1.3b", model_path: str | None = None, lora: str | None = None,
                 main_device: str = "cuda:0", text_device: str | None = None, vae_dtype=torch.float32):
        from diffusers import UniPCMultistepScheduler, WanPipeline
        info = MODELS[model]
        self.info = info
        path = model_path or info["repo"]
        if not os.path.isdir(path):
            from huggingface_hub import snapshot_download
            path = snapshot_download(path, allow_patterns=["model_index.json", "tokenizer/*", "text_encoder/*",
                                                           "vae/*", "transformer/*", "scheduler/*"])
        cuda = torch.cuda.is_available()
        self.main = torch.device(main_device if cuda else "cpu")
        text_device = text_device or ("cuda:1" if cuda and torch.cuda.device_count() > 1 else main_device)
        self.text_dev = torch.device(text_device if cuda else "cpu")
        self.dtype = torch.float16 if cuda else torch.float32
        if cuda:
            configure_hardware(Dist(device=self.main))
        self.tok, self.te = load_text_encoder(path, self.text_dev, self.dtype)
        vae = load_vae(path, self.main, vae_dtype if cuda else torch.float32)
        if hasattr(vae, "enable_tiling"):
            vae.enable_tiling()
        tr = load_transformer(path, self.main, self.dtype)
        sched = UniPCMultistepScheduler.from_pretrained(path, subfolder="scheduler")
        self.pipe = WanPipeline(tokenizer=None, text_encoder=None, vae=vae, transformer=tr,
                                scheduler=UniPCMultistepScheduler.from_config(sched.config, flow_shift=info["flow_shift"]),
                                expand_timesteps=info["expand_timesteps"])
        self.lora_loaded = False
        if lora:
            self.load_lora(lora)

    def load_lora(self, path: str, name: str = "afterframe") -> None:
        self.pipe.load_lora_weights(path, adapter_name=name)
        self.lora_name, self.lora_loaded = name, True

    def set_flow_shift(self, shift: float) -> None:
        from diffusers import UniPCMultistepScheduler
        self.pipe.scheduler = UniPCMultistepScheduler.from_config(self.pipe.scheduler.config, flow_shift=shift)

    @torch.no_grad()
    def __call__(self, prompt: str, negative_prompt: str = DEFAULT_NEGATIVE, seed: int = 0, width: int = 832,
                 height: int = 480, num_frames: int = 81, steps: int = 30, guidance: float = 5.0,
                 lora_strength: float | None = 1.0) -> np.ndarray:
        emb = encode_prompts(self.tok, self.te, [prompt, negative_prompt])
        if emb[0]["truncated"]:
            print("warning: prompt longer than 512 tokens; the end is truncated", flush=True)
        if self.lora_loaded:
            if lora_strength is None or lora_strength == 0:
                self.pipe.disable_lora()
            else:
                self.pipe.enable_lora()
                self.pipe.set_adapters([self.lora_name], [lora_strength])
        p = pad_text(emb[0]["embeds"]).unsqueeze(0).to(self.main, self.dtype)
        n = pad_text(emb[1]["embeds"]).unsqueeze(0).to(self.main, self.dtype)
        ctx = torch.autocast("cuda", dtype=torch.float16) if self.main.type == "cuda" else contextlib.nullcontext()
        with ctx:
            out = self.pipe(prompt_embeds=p, negative_prompt_embeds=n, height=height, width=width,
                            num_frames=num_frames, num_inference_steps=steps, guidance_scale=guidance,
                            generator=torch.Generator(self.main).manual_seed(seed), output_type="np")
        return out.frames[0]


def side_by_side(a: str, b: str, dst: str, labels=("base", "LoRA")) -> None:
    vf = (f"[0:v]drawtext=text='{labels[0]}':x=10:y=10:fontcolor=white:fontsize=20:box=1:boxcolor=black@0.5[a];"
          f"[1:v]drawtext=text='{labels[1]}':x=10:y=10:fontcolor=white:fontsize=20:box=1:boxcolor=black@0.5[b];"
          f"[a][b]hstack=inputs=2")
    r = subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", a, "-i", b, "-filter_complex", vf, "-c:v", "libx264",
                        "-crf", "18", "-pix_fmt", "yuv420p", dst], capture_output=True)
    if r.returncode != 0:   # drawtext needs fontconfig; fall back to an unlabeled stack
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", a, "-i", b, "-filter_complex", "hstack=inputs=2",
                        "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", dst], check=True)


def cmd_infer(args) -> None:
    g = Generator(args.model, args.model_path, args.lora, args.device, args.text_device)
    prompts = [args.prompt] if args.prompt else load_val_prompts(args.prompts_file)
    out = Path(args.out_dir)
    for i, prompt in enumerate(prompts):
        common = dict(seed=args.seed + i, width=args.width, height=args.height, num_frames=args.frames,
                      steps=args.steps, guidance=args.guidance, negative_prompt=args.negative_prompt)
        stem = f"{i:02d}_seed{args.seed + i}"
        made = {}
        if args.compare_base or not args.lora:
            t0 = time.time()
            write_mp4(g(prompt, lora_strength=0 if args.lora else None, **common), out / f"{stem}_base.mp4", g.info["fps"])
            made["base"] = time.time() - t0
        if args.lora:
            t0 = time.time()
            write_mp4(g(prompt, lora_strength=args.lora_strength, **common), out / f"{stem}_lora.mp4", g.info["fps"])
            made["lora"] = time.time() - t0
        if len(made) == 2:
            side_by_side(str(out / f"{stem}_base.mp4"), str(out / f"{stem}_lora.mp4"), str(out / f"{stem}_compare.mp4"))
        (out / f"{stem}.json").write_text(json.dumps({"prompt": prompt, **common, "lora": args.lora,
                                                      "lora_strength": args.lora_strength, "seconds": made}, indent=1))
        print(f"[{i}] {prompt[:70]}...  {made}", flush=True)


# ------------------------------------------------------------------------------------------------
def build_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)

    def common(p):
        p.add_argument("--preset", default="balanced", choices=sorted(PRESETS))
        p.add_argument("--stage", type=int, default=0, help="index into the preset's progressive stages")
        p.add_argument("--model", choices=sorted(MODELS), help="override the preset's base model (e.g. smoke-test wan22_5b)")
        p.add_argument("--model-path", help="local diffusers folder or HF repo for that model")
        p.add_argument("--cache-dir", default="/kaggle/temp/afterframe_cache")
        p.add_argument("--target-fps", type=float)
        p.add_argument("--clips-per-video", type=int)
        p.add_argument("--val-prompts", help="file with one validation prompt per line")

    c = sub.add_parser("cache")
    common(c)
    c.add_argument("--dataset", nargs="+", required=True, help="folders searched for shard_*/manifest.jsonl")
    c.add_argument("--shards", help="comma list (default: all found)")
    c.add_argument("--what", default="text,latents")
    c.add_argument("--max-videos-per-shard", type=int)
    c.add_argument("--text-batch", type=int, default=8)
    c.add_argument("--vae-dtype", choices=["auto", "fp16", "fp32"], default="auto")
    c.add_argument("--vae-fp16-tolerance", type=float, default=0.02)
    c.add_argument("--vae-tiling", action=argparse.BooleanOptionalAction, default=True)
    c.add_argument("--decode-workers", type=int, default=3)
    c.add_argument("--negative-prompt", default=DEFAULT_NEGATIVE)

    t = sub.add_parser("train")
    common(t)
    t.add_argument("--output-dir", default="/kaggle/working/afterframe_run")
    t.add_argument("--shards", help="comma list of cached shards to train on (default: all cached for the stage)")
    t.add_argument("--passes", type=int, default=1, help="passes over this chunk of shards before stopping")
    t.add_argument("--rank", type=int)
    t.add_argument("--alpha", type=float)
    t.add_argument("--targets", choices=sorted(TARGETS))
    t.add_argument("--lr", type=float)
    t.add_argument("--accum", type=int)
    t.add_argument("--batch", type=int)
    t.add_argument("--lora-dropout", type=float, default=0.0)
    t.add_argument("--optimizer", choices=["adamw8bit", "adamw"], default="adamw8bit")
    t.add_argument("--weight-decay", type=float, default=1e-4)
    t.add_argument("--lr-scheduler", choices=["cosine", "constant"], default="cosine")
    t.add_argument("--warmup-steps", type=int, default=50)
    t.add_argument("--total-steps", type=int, default=3000, help="horizon of the cosine schedule, across all sessions")
    t.add_argument("--min-lr-ratio", type=float, default=0.1)
    t.add_argument("--max-grad-norm", type=float, default=1.0)
    t.add_argument("--mixed-precision", choices=["fp16", "no", "bf16"], default="fp16")
    t.add_argument("--gradient-checkpointing", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--timestep-sampling", choices=["logit_normal", "uniform"], default="logit_normal")
    t.add_argument("--timestep-shift", type=float)
    t.add_argument("--seed", type=int, default=42)
    t.add_argument("--max-steps", type=int, help="stop at this global step")
    t.add_argument("--max-clips", type=int, help="use only the first N training clips (smoke tests)")
    t.add_argument("--max-train-minutes", type=float, help="stop and checkpoint after this many minutes")
    t.add_argument("--save-every", type=int, default=100)
    t.add_argument("--keep-checkpoints", type=int, default=3)
    t.add_argument("--resume", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--resume-from", nargs="*", help="extra folders searched for step_*/ checkpoints")
    t.add_argument("--hub-repo", help="optional private HF repo to mirror checkpoints to (needs HF_TOKEN)")
    t.add_argument("--log-every", type=int, default=5)
    t.add_argument("--val-loss-every", type=int, default=100)
    t.add_argument("--val-clips", type=int, default=24)
    t.add_argument("--val-every", type=int, default=300, help="sample videos every N steps (0 = off)")
    t.add_argument("--val-steps", type=int, default=20)
    t.add_argument("--val-frames", type=int)
    t.add_argument("--val-guidance", type=float, default=5.0)
    t.add_argument("--num-workers", type=int, default=2)

    e = sub.add_parser("export")
    e.add_argument("--checkpoint")
    e.add_argument("--checkpoint-roots", nargs="*")
    e.add_argument("--out", default="afterframe_sora2_lora.safetensors")

    i = sub.add_parser("infer")
    i.add_argument("--model", default="wan21_1.3b", choices=sorted(MODELS))
    i.add_argument("--model-path")
    i.add_argument("--lora")
    i.add_argument("--lora-strength", type=float, default=1.0)
    i.add_argument("--compare-base", action="store_true")
    i.add_argument("--prompt")
    i.add_argument("--prompts-file")
    i.add_argument("--negative-prompt", default=DEFAULT_NEGATIVE)
    i.add_argument("--seed", type=int, default=0)
    i.add_argument("--width", type=int, default=832)
    i.add_argument("--height", type=int, default=480)
    i.add_argument("--frames", type=int, default=49)
    i.add_argument("--steps", type=int, default=30)
    i.add_argument("--guidance", type=float, default=5.0)
    i.add_argument("--device", default="cuda:0")
    i.add_argument("--text-device")
    i.add_argument("--out-dir", default="/kaggle/working/inference")

    sub.add_parser("presets")
    return ap


def main(argv=None) -> None:
    args = build_parser().parse_args(argv)
    if args.cmd == "cache":
        args.what = set(args.what.split(","))
        cmd_cache(args)
    elif args.cmd == "train":
        cmd_train(args)
    elif args.cmd == "export":
        cmd_export(args)
    elif args.cmd == "infer":
        cmd_infer(args)
    elif args.cmd == "presets":
        print(json.dumps(PRESETS, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/extract_afterframe.py
#!/usr/bin/env python3
"""
extract_afterframe.py — resumable, rate-limited extractor for the Afterframe archive.

What the site exposes (found by reading /afterframe-sites.js and the inline app script, then
checking each endpoint by hand on 2026-11-20):

  POST /api/session                      -> {"id", "admin", "reviewRequired", ...}; sets the visitor cookie
  GET  /api/post-count                   -> {"posts": N}
  GET  /api/docs/posts?details=1&sort=created&direction=desc&limit<=1000[&after=<id>&afterCreated=<ms>]
                                         -> {"rows": [{"id", "data": {...}}], "cursor": {"id", "created"} | null}
  GET  /api/docs/posts/<id>?details=1    -> {"row": {...}}
  GET  /api/post-prompt/<id>             -> {"prompt": str | null}   ** OWNER ONLY **: 403 {"code":"not_granted"} for visitors
  GET  /media/<asset-uuid>               -> original uploaded bytes (video/mp4), supports HTTP Range

Every request carries the `X-Afterframe: 1` header the web client sends. All calls are GET, except
POST /api/session, which the site's own page makes on every load to create a visitor session. Nothing
is written to the site.

Stages (each one is resumable; state lives in --out/state):

  scan      page through the whole catalog -> state/catalog.jsonl
  prompts   fetch the stored generation prompt per post (needs the owner's signed-in cookie)
            -> state/prompts.jsonl
  download  download eligible posts at their original bytes, with range-resume, retries, sha256 and
            ffprobe validation -> raw/shard_XXX/<post_id>.<ext>, state/downloads.jsonl
  status    print counts

A post is ELIGIBLE for download/training only if ALL of these hold:
  * the backend returned a prompt string that is non-empty after whitespace stripping
    (it is never generated, inferred or rewritten);
  * the post passes the permission policy you chose (--permission-policy);
  * it is not hidden by moderation (reportCount below the site's hide threshold) and has a media asset.
"""
from __future__ import annotations

import argparse
import concurrent.futures as cf
import hashlib
import json
import os
import random
import re
import shutil
import subprocess
import sys
import threading
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Iterator

import requests

try:
    from tqdm import tqdm
except ImportError:  # progress bars are optional
    tqdm = None

BASE_URL = "https://afterframe.marcelalia2.chatgpt.site"
USER_AGENT = "afterframe-dataset-extractor/1.0 (+owner-authorised archive export)"
PAGE_LIMIT = 1000          # the site's own client never asks for more than 1000 rows per page
HIDE_AT_REPORTS = 3        # posts with this many reports are hidden from visitors in the web client
SORA_LAUNCH_SORA2_MS = 1759190400000  # 2025-09-30T00:00:00Z — Sora 2 public launch
SORA2_DIMS = {(1280, 704), (704, 1280), (1792, 1024), (1024, 1792), (1280, 720), (720, 1280),
              (1024, 1024), (704, 704)}


# --------------------------------------------------------------------------------------------
# small utilities
# --------------------------------------------------------------------------------------------
def log(msg: str) -> None:
    print(time.strftime("[%H:%M:%S] ") + msg, file=sys.stderr, flush=True)


def make_bar(**kw):
    """tqdm progress bar; in non-interactive logs it redraws at most every 30 s."""
    if tqdm is None:
        return None
    return tqdm(mininterval=0.5 if sys.stderr.isatty() else 30, **kw)


def read_jsonl(path: Path) -> Iterator[dict]:
    if not path.exists():
        return
    with path.open("r", encoding="utf-8") as fh:
        for line_no, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                # a crash mid-write can leave one truncated final line; ignore it, it will be redone
                log(f"warning: skipping unreadable line {line_no} in {path.name}")


class JsonlAppender:
    """Thread-safe append-only JSONL writer (fsync'd so a killed session loses at most one line)."""

    def __init__(self, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        self.fh = path.open("a", encoding="utf-8")
        self.lock = threading.Lock()

    def write(self, obj: dict) -> None:
        line = json.dumps(obj, ensure_ascii=False, sort_keys=True)
        with self.lock:
            self.fh.write(line + "\n")
            self.fh.flush()
            os.fsync(self.fh.fileno())

    def close(self) -> None:
        self.fh.close()


class RateLimiter:
    """Token bucket shared by all threads: at most `rate` requests per second, burst `burst`."""

    def __init__(self, rate: float, burst: int = 2):
        self.rate, self.capacity = rate, float(burst)
        self.tokens, self.last = float(burst), time.monotonic()
        self.lock = threading.Lock()

    def acquire(self) -> None:
        while True:
            with self.lock:
                now = time.monotonic()
                self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
                self.last = now
                if self.tokens >= 1:
                    self.tokens -= 1
                    return
                wait = (1 - self.tokens) / self.rate
            time.sleep(wait)


def crockford_ulid_ms(ulid: str) -> int | None:
    """Decode the 48-bit millisecond timestamp at the start of a ULID (Sora generation ids are ULIDs)."""
    alphabet = "0123456789ABCDEFGHJKMNPQRSTVWXYZ"
    head = ulid[:10].upper()
    if len(head) != 10 or any(c not in alphabet for c in head) or head[0] > "7":
        return None
    value = 0
    for c in head:
        value = value * 32 + alphabet.index(c)
    # 2023-01-01 .. 2030-01-01: anything outside is not a real generation timestamp
    return value if 1672531200000 <= value <= 1893456000000 else None


def sora_generation_id(name: str | None) -> str | None:
    m = re.search(r"(gen_[0-9a-z]{26})", name or "", re.I)
    return m.group(1).lower() if m else None


def shard_of(post_id: str, num_shards: int) -> int:
    return int(hashlib.sha1(post_id.encode()).hexdigest(), 16) % num_shards


# --------------------------------------------------------------------------------------------
# HTTP client
# --------------------------------------------------------------------------------------------
class ApiError(Exception):
    def __init__(self, status: int, payload: Any):
        super().__init__(f"HTTP {status}: {payload}")
        self.status, self.payload = status, payload

    @property
    def code(self) -> str | None:
        return self.payload.get("code") if isinstance(self.payload, dict) else None


class AfterframeClient:
    RETRY_STATUS = {408, 425, 429, 500, 502, 503, 504, 520, 521, 522, 523, 524}

    def __init__(self, base_url: str = BASE_URL, cookie: str | None = None, rate: float = 3.0,
                 max_retries: int = 6, timeout: float = 60.0):
        self.base = base_url.rstrip("/")
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": USER_AGENT, "X-Afterframe": "1", "Accept": "application/json"})
        if cookie:
            # the owner's browser cookie (copied from devtools); sent verbatim, never logged
            self.session.headers["Cookie"] = cookie.strip()
        self.limiter = RateLimiter(rate, burst=max(1, int(rate)))
        self.max_retries, self.timeout = max_retries, timeout
        self.session_info: dict | None = None

    def _request(self, method: str, path: str, *, stream: bool = False, headers: dict | None = None,
                 accept_status: Iterable[int] = (200,)) -> requests.Response:
        url = self.base + path
        accept_status = set(accept_status)
        for attempt in range(self.max_retries + 1):
            self.limiter.acquire()
            try:
                resp = self.session.request(method, url, stream=stream, timeout=self.timeout, headers=headers)
            except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as exc:
                if attempt == self.max_retries:
                    raise
                delay = min(60, 2 ** attempt) + random.random()
                log(f"network error on {method} {path} ({type(exc).__name__}); retry {attempt + 1} in {delay:.1f}s")
                time.sleep(delay)
                continue
            if resp.status_code in accept_status:
                return resp
            if resp.status_code in self.RETRY_STATUS and attempt < self.max_retries:
                retry_after = resp.headers.get("Retry-After")
                delay = float(retry_after) if retry_after and retry_after.isdigit() else min(120, 2 ** attempt) + random.random()
                log(f"HTTP {resp.status_code} on {method} {path}; retry {attempt + 1} in {delay:.1f}s")
                resp.close()
                time.sleep(delay)
                continue
            try:
                payload = resp.json()
            except ValueError:
                payload = resp.text[:300]
            raise ApiError(resp.status_code, payload)
        raise RuntimeError("unreachable")

    def api(self, method: str, path: str) -> Any:
        return self._request(method, "/api/" + path.lstrip("/")).json()

    # ---- endpoints -----------------------------------------------------------------------
    def start_session(self) -> dict:
        self.session_info = self.api("POST", "session")
        return self.session_info

    def post_count(self) -> int:
        return int(self.api("GET", "post-count")["posts"])

    def posts_page(self, cursor: dict | None, limit: int = PAGE_LIMIT) -> dict:
        params = {"details": "1", "sort": "created", "direction": "desc", "limit": str(limit)}
        if cursor:
            params.update({"after": cursor["id"], "afterCreated": str(cursor["created"])})
        return self.api("GET", "docs/posts?" + requests.compat.urlencode(params))

    def post_prompt(self, post_id: str) -> dict:
        return self.api("GET", "post-prompt/" + requests.utils.quote(post_id, safe=""))

    def media(self, asset_id: str, start: int = 0) -> requests.Response:
        headers = {"Accept": "*/*"}
        if start:
            headers["Range"] = f"bytes={start}-"
        return self._request("GET", "/media/" + requests.utils.quote(asset_id, safe=""), stream=True,
                             headers=headers, accept_status=(200, 206))


# --------------------------------------------------------------------------------------------
# stage 1: catalog scan
# --------------------------------------------------------------------------------------------
def stage_scan(client: AfterframeClient, out: Path, limit: int, max_pages: int | None) -> None:
    state_dir = out / "state"
    state_dir.mkdir(parents=True, exist_ok=True)
    catalog_path, scan_state_path = state_dir / "catalog.jsonl", state_dir / "scan_state.json"
    seen = {row["id"] for row in read_jsonl(catalog_path)}
    state = json.loads(scan_state_path.read_text()) if scan_state_path.exists() else {}
    if state.get("complete"):
        # A finished scan is refreshed from the top: new posts appear first in created-desc order,
        # so we stop as soon as a whole page is already known.
        log(f"catalog already complete ({len(seen)} posts); checking for new posts")
        state = {"cursor": None, "complete": False, "refresh": True}
    cursor = state.get("cursor")
    expected = client.post_count()
    log(f"server reports {expected} posts; {len(seen)} already in catalog; resuming from cursor {cursor}")
    writer = JsonlAppender(catalog_path)
    pages = 0
    bar = make_bar(total=expected, initial=len(seen), unit="post", desc="scan")
    try:
        while True:
            page = client.posts_page(cursor, limit)
            rows = page.get("rows") or []
            new = 0
            fetched_at = int(time.time() * 1000)
            for row in rows:
                if row["id"] in seen:
                    continue
                seen.add(row["id"])
                new += 1
                writer.write({"id": row["id"], "data": row.get("data") or {}, "fetched_at": fetched_at})
            if bar:
                bar.update(new)
            pages += 1
            cursor = page.get("cursor")
            done = not cursor or not rows or (state.get("refresh") and new == 0)
            state = {"cursor": cursor, "complete": bool(done), "pages_this_run": pages,
                     "posts": len(seen), "server_count": expected, "updated": fetched_at,
                     "refresh": state.get("refresh", False)}
            tmp = scan_state_path.with_suffix(".tmp")
            tmp.write_text(json.dumps(state, indent=1))
            tmp.replace(scan_state_path)
            if done or (max_pages and pages >= max_pages):
                break
    finally:
        writer.close()
        if bar:
            bar.close()
    log(f"scan {'complete' if state['complete'] else 'paused'}: {len(seen)} posts in catalog "
        f"(server count {expected}; the count includes posts hidden from visitors, so they can differ)")


def load_catalog(out: Path) -> dict[str, dict]:
    catalog: dict[str, dict] = {}
    for row in read_jsonl(out / "state" / "catalog.jsonl"):
        catalog[row["id"]] = row  # later lines win
    return catalog


# --------------------------------------------------------------------------------------------
# stage 2: prompts (owner only)
# --------------------------------------------------------------------------------------------
def require_owner(client: AfterframeClient, out: Path | None = None, probe_id: str | None = None) -> None:
    info = client.start_session()
    if out is not None:
        rec = {"checked_at": int(time.time() * 1000), "session_admin": bool(info.get("admin"))}
        if not info.get("admin") and probe_id:
            try:
                client.post_prompt(probe_id)
                rec["probe"] = {"post": probe_id, "http": 200}
            except ApiError as exc:
                rec["probe"] = {"post": probe_id, "http": exc.status, "code": exc.code}
        (out / "state").mkdir(parents=True, exist_ok=True)
        (out / "state" / "prompt_access.json").write_text(json.dumps(rec, indent=1))
    if not info.get("admin"):
        raise SystemExit(
            "The prompt endpoint (/api/post-prompt/<id>) is owner-only and this session is not the owner "
            "(session.admin is false).\n"
            "Sign in as the owner in a browser (Afterframe > Settings > Owner sign-in), copy the request "
            "'Cookie' header for afterframe.marcelalia2.chatgpt.site from devtools, and pass it with "
            "--cookie-file or the AFTERFRAME_COOKIE environment variable. The extractor will not try to "
            "get prompts any other way.")
    log("owner session confirmed (session.admin is true)")


def stage_prompts(client: AfterframeClient, out: Path, workers: int, refetch_errors: bool) -> None:
    catalog = load_catalog(out)
    require_owner(client, out, next(iter(catalog), None))
    prompts_path = out / "state" / "prompts.jsonl"
    done: dict[str, dict] = {r["id"]: r for r in read_jsonl(prompts_path)}
    todo = [pid for pid in catalog
            if pid not in done or (refetch_errors and done[pid].get("status") == "error")]
    log(f"{len(catalog)} catalog posts; {len(done)} prompt lookups recorded; {len(todo)} to fetch")
    writer = JsonlAppender(prompts_path)

    def fetch(pid: str) -> dict:
        try:
            payload = client.post_prompt(pid)
        except ApiError as exc:
            if exc.code == "not_granted":
                raise SystemExit("owner access was lost mid-run (403 not_granted); refresh the cookie and rerun")
            status = "missing" if exc.status == 404 else "error"
            return {"id": pid, "status": status, "http": exc.status, "error": str(exc.payload)[:300]}
        prompt = payload.get("prompt") if isinstance(payload, dict) else None
        has = isinstance(prompt, str) and prompt.strip() != ""
        rec = {"id": pid, "status": "ok" if has else "empty", "prompt": prompt if isinstance(prompt, str) else None,
               "fetched_at": int(time.time() * 1000)}
        extra = {k: v for k, v in (payload.items() if isinstance(payload, dict) else []) if k != "prompt"}
        if extra:
            rec["extra"] = extra  # keep any other fields the backend returns (e.g. model) verbatim
        return rec

    bar = make_bar(total=len(todo), unit="post", desc="prompts")
    counts: dict[str, int] = {}
    try:
        with cf.ThreadPoolExecutor(max_workers=workers) as pool:
            for rec in pool.map(fetch, todo):
                writer.write(rec)
                counts[rec["status"]] = counts.get(rec["status"], 0) + 1
                if bar:
                    bar.update(1)
                    bar.set_postfix(counts)
    finally:
        writer.close()
        if bar:
            bar.close()
    log(f"prompt fetch finished: {counts}")


def load_prompts(out: Path) -> dict[str, dict]:
    return {r["id"]: r for r in read_jsonl(out / "state" / "prompts.jsonl")}


# --------------------------------------------------------------------------------------------
# permission policy
# --------------------------------------------------------------------------------------------
@dataclass
class PermissionPolicy:
    mode: str                         # "allowlist" | "owner-uploads" | "all-attested" | "pending"
    uploaders: set[str]

    @classmethod
    def from_args(cls, args, client: AfterframeClient | None) -> "PermissionPolicy":
        mode = args.permission_policy
        if mode is None:
            raise SystemExit(
                "Choose which posts you have permission to train on with --permission-policy:\n"
                "  owner-uploads  only posts uploaded by the owner's own visitor profile(s) (--uploader-id)\n"
                "  allowlist      only uploader ids listed in --uploaders-file (one 64-hex id per line)\n"
                "  all-attested   every post; you attest you hold training rights for all of them\n"
                "Afterframe accepts uploads from any visitor, so posts are not all yours by default.")
        if mode == "pending":            # permissions not decided yet: nothing is eligible
            return cls(mode, set())
        ids: set[str] = set(args.uploader_id or [])
        if mode == "allowlist":
            if not args.uploaders_file:
                raise SystemExit("--permission-policy allowlist needs --uploaders-file")
            ids |= {ln.strip().lower() for ln in Path(args.uploaders_file).read_text().splitlines()
                    if re.fullmatch(r"[0-9a-fA-F]{64}", ln.strip())}
        if mode == "owner-uploads" and not ids:
            if client is None or client.session_info is None:
                raise SystemExit("owner-uploads needs --uploader-id or an owner session")
            ids.add(client.session_info["id"])
        if mode in ("allowlist", "owner-uploads") and not ids:
            raise SystemExit("the permission policy resolved to an empty uploader list")
        return cls(mode, {i.lower() for i in ids})

    def allows(self, data: dict) -> bool:
        if self.mode == "pending":
            return False
        return self.mode == "all-attested" or str(data.get("by", "")).lower() in self.uploaders


def eligibility(post: dict, prompt_rec: dict | None, policy: PermissionPolicy | None) -> tuple[bool, str]:
    data = post.get("data") or {}
    if not data.get("asset"):
        return False, "no_media_asset"
    if (data.get("reportCount") or 0) >= HIDE_AT_REPORTS:
        return False, "hidden_by_moderation"
    if prompt_rec is None:
        return False, "prompt_not_fetched"   # see state/prompt_access.json for whether access was denied
    if prompt_rec.get("status") != "ok":
        return False, "prompt_" + str(prompt_rec.get("status"))
    if policy is not None and not policy.allows(data):
        return False, "no_permission"
    return True, "eligible"


# --------------------------------------------------------------------------------------------
# provenance
# --------------------------------------------------------------------------------------------
def c2pa_scan(path: Path, max_bytes: int = 4 << 20) -> dict:
    """Look for an embedded C2PA manifest and pull out its claims without validating signatures.

    Sora exports carry a JUMBF/C2PA manifest in a top-level `uuid` box before `moov`. We record the
    softwareAgent, digitalSourceType and signer organisation strings. This is evidence, not
    cryptographic verification: run `c2patool <file>` for a validated report.
    """
    with path.open("rb") as fh:
        head = fh.read(max_bytes)
    res: dict[str, Any] = {"present": b"jumdc2pa" in head or b"c2pa.claim" in head}
    if not res["present"]:
        return res
    m = re.search(rb"softwareAgent.{1,4}dname.([A-Za-z0-9 ._\-]{1,40})", head, re.S)
    if m:
        res["software_agent"] = m.group(1).decode("latin1").strip()
    m = re.search(rb"digitalsourcetype/(compositeWithTrainedAlgorithmicMedia|trainedAlgorithmicMedia|"
                  rb"compositeSynthetic|algorithmicMedia|digitalCapture|composite|minorHumanEdits|humanEdits)", head)
    if m:
        res["digital_source_type"] = m.group(1).decode()
    orgs = sorted({o.decode("latin1") for o in re.findall(rb"\x06\x03U\x04\n\x0c.(OpenAI)", head, re.S)})
    if orgs:
        res["signer_org"] = orgs[0]
    m = re.search(rb"\x06\x03U\x04\x03\x0c.([ -~]{4,64}?)0", head, re.S)
    if m and b"Sora" in m.group(1):
        res["signer_cn"] = m.group(1).decode("latin1")
    return res


def provenance_assessment(data: dict, probe: dict | None, c2pa: dict | None, prompt_rec: dict | None) -> dict:
    """Combine every signal we have about which model made the video. Never upgrades to 'confirmed'
    unless the backend itself names Sora 2."""
    signals: dict[str, Any] = {}
    gen_id = sora_generation_id(data.get("name"))
    signals["sora_generation_id"] = gen_id
    ts = crockford_ulid_ms(gen_id[4:]) if gen_id else None
    signals["generation_time_ms"] = ts
    signals["after_sora2_launch"] = None if ts is None else ts >= SORA_LAUNCH_SORA2_MS
    if c2pa:
        signals["c2pa_present"] = c2pa.get("present", False)
        signals["c2pa_software_agent"] = c2pa.get("software_agent")
        signals["c2pa_signer_org"] = c2pa.get("signer_org")
        signals["c2pa_source_type"] = c2pa.get("digital_source_type")
    if probe:
        signals["has_audio"] = probe.get("has_audio")
        signals["sora2_resolution"] = (probe.get("width"), probe.get("height")) in SORA2_DIMS
    backend_model = None
    if prompt_rec and isinstance(prompt_rec.get("extra"), dict):
        for key in ("model", "generationModel", "generator"):
            if prompt_rec["extra"].get(key):
                backend_model = str(prompt_rec["extra"][key])
    signals["backend_model"] = backend_model

    if backend_model:
        label = "sora2_confirmed_by_backend" if re.search(r"sora[\s_-]*2", backend_model, re.I) else "other_model_by_backend"
    else:
        is_sora = signals.get("c2pa_software_agent") == "Sora" and signals.get("c2pa_signer_org") == "OpenAI"
        strong = [signals.get("after_sora2_launch") is True, signals.get("has_audio") is True,
                  signals.get("sora2_resolution") is True]
        if is_sora and all(strong):
            label = "sora2_likely"          # made by Sora (C2PA), with every Sora-2-only trait present
        elif is_sora:
            label = "sora_version_uncertain"
        elif c2pa is None and probe is None:
            label = "unverified_not_downloaded"
        else:
            label = "provenance_uncertain"  # no OpenAI/Sora C2PA claim found
    return {"label": label, "signals": signals}


# --------------------------------------------------------------------------------------------
# stage 3: download
# --------------------------------------------------------------------------------------------
def ffprobe(path: Path) -> dict:
    cmd = ["ffprobe", "-v", "error", "-show_format", "-show_streams", "-of", "json", str(path)]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if proc.returncode != 0:
        return {"ok": False, "error": proc.stderr.strip()[:300]}
    info = json.loads(proc.stdout or "{}")
    vids = [s for s in info.get("streams", []) if s.get("codec_type") == "video"]
    auds = [s for s in info.get("streams", []) if s.get("codec_type") == "audio"]
    if not vids:
        return {"ok": False, "error": "no video stream"}
    v = vids[0]

    def rate(txt: str | None) -> float | None:
        if not txt or txt in ("0/0",):
            return None
        num, _, den = txt.partition("/")
        return float(num) / float(den or 1) if float(den or 1) else None

    width, height = int(v.get("width") or 0), int(v.get("height") or 0)
    rot = 0
    for sd in v.get("side_data_list") or []:
        if "rotation" in sd:
            rot = int(sd["rotation"])
    if abs(rot) in (90, 270):
        width, height = height, width
    duration = float(v.get("duration") or info.get("format", {}).get("duration") or 0)
    fps = rate(v.get("avg_frame_rate")) or rate(v.get("r_frame_rate"))
    nb = int(v["nb_frames"]) if str(v.get("nb_frames", "")).isdigit() else (round(duration * fps) if fps else None)
    ok = width > 0 and height > 0 and duration > 0 and bool(fps)
    return {"ok": ok, "error": None if ok else "bad stream parameters", "width": width, "height": height,
            "fps": fps, "r_frame_rate": v.get("r_frame_rate"), "avg_frame_rate": v.get("avg_frame_rate"),
            "duration": duration, "nb_frames": nb, "vcodec": v.get("codec_name"), "pix_fmt": v.get("pix_fmt"),
            "rotation": rot, "has_audio": bool(auds), "acodec": auds[0].get("codec_name") if auds else None,
            "container": info.get("format", {}).get("format_name"),
            "bit_rate": int(info.get("format", {}).get("bit_rate") or 0)}


def deep_decode_ok(path: Path) -> tuple[bool, str]:
    """Decode every video frame; catches truncated or corrupt streams that ffprobe accepts."""
    proc = subprocess.run(["ffmpeg", "-v", "error", "-xerror", "-i", str(path), "-map", "0:v:0", "-f", "null", "-"],
                          capture_output=True, text=True, timeout=600)
    return proc.returncode == 0 and not proc.stderr.strip(), proc.stderr.strip()[:300]


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def download_one(client: AfterframeClient, post: dict, dest_dir: Path, deep_verify: bool) -> dict:
    pid, data = post["id"], post["data"]
    ext = ".mov" if "quicktime" in str(data.get("type", "")) else ".mp4"
    final = dest_dir / f"{pid}{ext}"
    part = final.with_suffix(final.suffix + ".part")
    dest_dir.mkdir(parents=True, exist_ok=True)
    total = None
    for attempt in range(4):
        start = part.stat().st_size if part.exists() else 0
        try:
            resp = client.media(data["asset"], start)
        except ApiError as exc:
            return {"id": pid, "status": "http_error", "http": exc.status, "error": str(exc.payload)[:200]}
        with resp:
            if start and resp.status_code == 200:      # server ignored Range: start over
                start = 0
            cr = resp.headers.get("Content-Range")
            if cr and "/" in cr:
                total = int(cr.rsplit("/", 1)[1])
            elif resp.headers.get("Content-Length"):
                total = start + int(resp.headers["Content-Length"])
            try:
                with part.open("ab" if start else "wb") as fh:
                    for chunk in resp.iter_content(1 << 20):
                        fh.write(chunk)
            except (requests.ConnectionError, requests.exceptions.ChunkedEncodingError, requests.Timeout):
                time.sleep(2 ** attempt)
                continue                                # resume from the partial file
        size = part.stat().st_size
        if total is not None and size != total:
            if size > total:
                part.unlink()
            continue
        break
    else:
        return {"id": pid, "status": "incomplete", "bytes": part.stat().st_size if part.exists() else 0, "expected": total}

    probe = ffprobe(part)
    if not probe["ok"]:
        bad = dest_dir.parent.parent / "quarantine" / part.name.replace(".part", "")
        bad.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(part), bad)
        return {"id": pid, "status": "corrupt", "error": probe["error"], "quarantined": str(bad)}
    if deep_verify:
        ok, err = deep_decode_ok(part)
        if not ok:
            bad = dest_dir.parent.parent / "quarantine" / part.name.replace(".part", "")
            bad.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(part), bad)
            return {"id": pid, "status": "corrupt", "error": "decode: " + err, "quarantined": str(bad)}
    digest = sha256_file(part)
    c2pa = c2pa_scan(part)
    part.replace(final)
    return {"id": pid, "status": "ok", "path": str(final), "bytes": final.stat().st_size, "sha256": digest,
            "probe": probe, "c2pa": c2pa, "asset": data["asset"], "downloaded_at": int(time.time() * 1000)}


def stage_download(client: AfterframeClient, out: Path, args) -> None:
    catalog, prompts = load_catalog(out), load_prompts(out)
    if args.test_sample:
        # Pipeline test mode: downloads a few posts regardless of prompt so the downloader, ffprobe,
        # dedup and provenance code can be exercised. They go to test_sample/, never into raw/, and
        # prepare_dataset.py never reads that folder.
        policy = None
        posts = [p for p in catalog.values() if p["data"].get("asset")]
        random.Random(0).shuffle(posts)
        selected = posts[: args.test_sample]
        root = out / "test_sample"
        dl_path = out / "state" / "test_sample_downloads.jsonl"
    else:
        if client.session_info is None:
            client.start_session()
        policy = PermissionPolicy.from_args(args, client)
        selected, reasons = [], {}
        for pid, post in catalog.items():
            ok, why = eligibility(post, prompts.get(pid), policy)
            reasons[why] = reasons.get(why, 0) + 1
            if ok:
                selected.append(post)
        log(f"eligibility over {len(catalog)} catalog posts: {reasons}")
        root = out / "raw"
        dl_path = out / "state" / "downloads.jsonl"

    if args.num_shards > 1:
        wanted = set(range(args.num_shards)) if args.shards is None else {int(s) for s in args.shards.split(",")}
        selected = [p for p in selected if shard_of(p["id"], args.num_shards) in wanted]

    done = {r["id"]: r for r in read_jsonl(dl_path)}
    ok_ids = {pid for pid, r in done.items() if r.get("status") == "ok" and Path(r["path"]).exists()}
    seen_assets = {done[pid]["asset"] for pid in ok_ids if done[pid].get("asset")}
    seen_hashes = {done[pid]["sha256"]: pid for pid in ok_ids}
    queue, dup_assets = [], 0
    for post in sorted(selected, key=lambda p: p["data"].get("created", 0)):
        if post["id"] in ok_ids or done.get(post["id"], {}).get("status") in ("corrupt", "duplicate"):
            continue
        if post["data"]["asset"] in seen_assets:      # same stored file referenced twice
            dup_assets += 1
            continue
        seen_assets.add(post["data"]["asset"])
        queue.append(post)
    budget = int(args.max_gb * 1e9) if args.max_gb else None
    log(f"{len(selected)} selected, {len(ok_ids)} already downloaded, {dup_assets} duplicate asset refs skipped, "
        f"{len(queue)} queued" + (f", budget {args.max_gb} GB" if budget else ""))
    if args.limit:
        queue = queue[: args.limit]

    writer = JsonlAppender(dl_path)
    done_paths = {pid: done[pid]["path"] for pid in ok_ids}
    created = lambda pid: (catalog[pid]["data"].get("created", 0), pid)  # noqa: E731
    used = 0
    lock = threading.Lock()
    stop = threading.Event()
    counts: dict[str, int] = {}
    bar = make_bar(total=len(queue), unit="video", desc="download")

    def work(post: dict) -> dict | None:
        if stop.is_set():
            return None
        shard = shard_of(post["id"], max(1, args.num_shards))
        return download_one(client, post, root / f"shard_{shard:03d}", args.deep_verify)

    try:
        with cf.ThreadPoolExecutor(max_workers=args.workers) as pool:
            futures = [pool.submit(work, p) for p in queue]
            for fut in cf.as_completed(futures):
                rec = fut.result()
                if rec is None:
                    continue
                if rec["status"] == "ok":
                    with lock:
                        prior = seen_hashes.get(rec["sha256"])
                        if prior and prior != rec["id"]:
                            # byte-identical re-upload: keep the earliest upload, whichever finished first
                            if created(rec["id"]) < created(prior):
                                Path(done_paths[prior]).unlink(missing_ok=True)
                                writer.write({"id": prior, "status": "duplicate", "duplicate_of": rec["id"],
                                              "sha256": rec["sha256"]})
                                counts["duplicate"] = counts.get("duplicate", 0) + 1
                                counts["ok"] -= 1
                                seen_hashes[rec["sha256"]] = rec["id"]
                                done_paths[rec["id"]] = rec["path"]
                            else:
                                Path(rec["path"]).unlink(missing_ok=True)
                                rec = {"id": rec["id"], "status": "duplicate", "duplicate_of": prior,
                                       "sha256": rec["sha256"]}
                        else:
                            seen_hashes[rec["sha256"]] = rec["id"]
                            done_paths[rec["id"]] = rec["path"]
                            used += rec["bytes"]
                            if budget and used >= budget:
                                stop.set()
                writer.write(rec)
                counts[rec["status"]] = counts.get(rec["status"], 0) + 1
                if bar:
                    bar.update(1)
                    bar.set_postfix(counts)
    finally:
        writer.close()
        if bar:
            bar.close()
    log(f"download finished: {counts}; {used / 1e9:.2f} GB this run" + (" (budget reached)" if stop.is_set() else ""))


# --------------------------------------------------------------------------------------------
# status
# --------------------------------------------------------------------------------------------
def stage_status(out: Path) -> None:
    catalog, prompts = load_catalog(out), load_prompts(out)
    downloads = {r["id"]: r for r in read_jsonl(out / "state" / "downloads.jsonl")}
    scan = json.loads((out / "state" / "scan_state.json").read_text()) if (out / "state" / "scan_state.json").exists() else {}
    reasons: dict[str, int] = {}
    for pid, post in catalog.items():
        _, why = eligibility(post, prompts.get(pid), None)
        reasons[why] = reasons.get(why, 0) + 1
    dl: dict[str, int] = {}
    for r in downloads.values():
        dl[r["status"]] = dl.get(r["status"], 0) + 1
    print(json.dumps({"catalog_posts": len(catalog), "scan": scan,
                      "prompt_lookups": len(prompts), "eligibility_before_permission_policy": reasons,
                      "downloads": dl}, indent=2))


# --------------------------------------------------------------------------------------------
def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["scan", "prompts", "download", "all", "status"])
    ap.add_argument("--out", type=Path, default=Path("afterframe_data"))
    ap.add_argument("--base-url", default=BASE_URL)
    ap.add_argument("--cookie-file", type=Path, help="file holding the owner's Cookie header (or set AFTERFRAME_COOKIE)")
    ap.add_argument("--rate", type=float, default=3.0, help="max HTTP requests per second, all threads together")
    ap.add_argument("--workers", type=int, default=4, help="concurrent prompt/download workers")
    ap.add_argument("--page-limit", type=int, default=PAGE_LIMIT)
    ap.add_argument("--max-pages", type=int, help="stop the scan after N pages (it resumes next run)")
    ap.add_argument("--refetch-errors", action="store_true")
    ap.add_argument("--permission-policy", choices=["owner-uploads", "allowlist", "all-attested", "pending"])
    ap.add_argument("--uploader-id", action="append", help="64-hex visitor id whose uploads you may use (repeatable)")
    ap.add_argument("--uploaders-file", help="file with one permitted 64-hex uploader id per line")
    ap.add_argument("--num-shards", type=int, default=16, help="posts are hashed into this many shards")
    ap.add_argument("--shards", help="comma-separated shard indexes to download this run (default all)")
    ap.add_argument("--max-gb", type=float, help="stop downloading after this many GB (incremental runs)")
    ap.add_argument("--limit", type=int, help="download at most N videos this run")
    ap.add_argument("--deep-verify", action="store_true", help="decode every frame to catch corruption")
    ap.add_argument("--test-sample", type=int, help="download N random posts to test_sample/ for pipeline tests only")
    args = ap.parse_args(argv)

    cookie = os.environ.get("AFTERFRAME_COOKIE")
    if args.cookie_file:
        cookie = args.cookie_file.read_text().strip()
    client = AfterframeClient(args.base_url, cookie=cookie, rate=args.rate)
    args.out.mkdir(parents=True, exist_ok=True)

    if args.stage in ("scan", "all"):
        client.start_session()
        stage_scan(client, args.out, args.page_limit, args.max_pages)
    if args.stage in ("prompts", "all"):
        stage_prompts(client, args.out, args.workers, args.refetch_errors)
    if args.stage in ("download", "all"):
        stage_download(client, args.out, args)
    if args.stage == "status":
        stage_status(args.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/prepare_dataset.py
#!/usr/bin/env python3
"""
prepare_dataset.py — turn an extraction folder (from extract_afterframe.py) into a training dataset.

  afterframe_data/                        (input; never modified)
    state/catalog.jsonl, prompts.jsonl, downloads.jsonl
    raw/shard_XXX/<post_id>.mp4           <- the ORIGINAL dataset; left untouched

  dataset/                                (output)
    dataset_manifest.jsonl                one JSON record per eligible video (first line is a header)
    shard_XXX/videos/<post_id>.mp4        hard link (or copy) of the original file, bytes unchanged
    shard_XXX/captions/<post_id>.txt      UTF-8: "Sora 2 video, prompt: <backend prompt>" (no trailing newline)
    shard_XXX/manifest.jsonl              the manifest rows of that shard (each shard is self-contained,
                                          so it can be uploaded or loaded as its own Kaggle dataset)
    splits/train.txt, splits/val.txt      post ids
    report.json                           counts, exclusions, provenance flags, duplicate groups

Training clips (resized, cropped, frame-resampled) are made later, in the notebook, as cached latents.
They never overwrite these files.

Rules this script enforces:
  * Only posts whose backend prompt is a non-empty string after whitespace stripping are included.
    The prompt is copied byte for byte; nothing is generated, inferred, summarised or rewritten.
  * The caption is exactly CAPTION_PREFIX + backend prompt.
  * Exact duplicates (same sha256 or same Sora generation id) and near-duplicates (perceptual hash of
    sampled frames) are grouped; one representative per group is kept (--keep-near-duplicates
    keeps them all), and every group sits wholly in train or wholly in val.
  * Provenance is assessed per video and flagged; it is never silently upgraded.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from extract_afterframe import (BASE_URL, HIDE_AT_REPORTS, PermissionPolicy, eligibility, load_catalog,  # noqa: E402
                                load_prompts, provenance_assessment, read_jsonl, sora_generation_id)

CAPTION_PREFIX = "Sora 2 video, prompt: "
MANIFEST_VERSION = 1


def make_caption(backend_prompt: str) -> str:
    if not isinstance(backend_prompt, str) or not backend_prompt.strip():
        raise ValueError("refusing to build a caption without a non-empty backend prompt")
    caption = CAPTION_PREFIX + backend_prompt
    assert caption.startswith(CAPTION_PREFIX) and caption[len(CAPTION_PREFIX):] == backend_prompt
    return caption


# --------------------------------------------------------------------------------------------
# perceptual signature for near-duplicate detection
# --------------------------------------------------------------------------------------------
SIG_FPS = 4          # signature frames per second of video
SIG_MAX_FRAMES = 64
SIG_DARK_MIN = 4     # quality flag: at least this many dark sampled frames *and* half of them


def _dhash_bits(img32: np.ndarray) -> int:
    rows8 = img32.reshape(8, 4, 32).mean(1)                              # 8 x 32
    xs = np.linspace(0, 31, 9)
    grid = np.stack([np.interp(xs, np.arange(32), r) for r in rows8])   # 8 x 9
    bits = (grid[:, 1:] > grid[:, :-1]).flatten()                      # 64 horizontal-gradient bits
    return int("".join("1" if b else "0" for b in bits), 2)


def frame_signature(path: Path, duration: float) -> dict | None:
    """One decode pass: frames sampled at SIG_FPS (capped), grayscale 32x32 -> 64-bit dHash per frame,
    plus motion / brightness statistics for quality flags. None if decoding fails."""
    fps = min(SIG_FPS, SIG_MAX_FRAMES / max(duration, 0.25))
    cmd = ["ffmpeg", "-v", "error", "-i", str(path), "-map", "0:v:0",
           "-vf", f"fps={fps:.5f},scale=32:32:flags=area,format=gray", "-f", "rawvideo", "-"]
    proc = subprocess.run(cmd, capture_output=True, timeout=600)
    n = len(proc.stdout) // 1024
    if proc.returncode != 0 or n == 0:
        return None
    thumbs = np.frombuffer(proc.stdout[: n * 1024], np.uint8).reshape(n, 32, 32).astype(np.float32)
    motion = float(np.abs(np.diff(thumbs, axis=0)).mean()) * fps if n > 1 else 0.0  # per second
    dark = int((thumbs.mean((1, 2)) < 12).sum())
    return {"dhash": [f"{_dhash_bits(t):016x}" for t in thumbs], "fps": round(fps, 5),
            "motion": round(motion, 3), "brightness": round(float(thumbs.mean()), 2),
            "dark_fraction": round(dark / n, 3)}


def popcount64(x: np.ndarray) -> np.ndarray:
    """Per-element popcount of a uint64 array (numpy>=2 has np.bitwise_count; older numpy uses unpackbits)."""
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    x = np.ascontiguousarray(x, dtype=np.uint64)
    return np.unpackbits(x.view(np.uint8).reshape(*x.shape, 8), axis=-1).sum(-1)


def near_duplicate_pairs(ids: list[str], sigs: dict[str, dict], frame_bits: int = 10,
                         min_match_fraction: float = 0.6) -> list[tuple[str, str, float]]:
    """Pairs of near-identical videos, tolerant of re-encoding, rescaling, trimming and time offsets.

    A frame of A "matches" B when some frame of B is within `frame_bits` (of 64) Hamming distance.
    A and B are near-identical when at least `min_match_fraction` of the frames of either one match
    the other (so a clip cut from a longer video counts). Candidate pairs come from 4 x 16-bit band
    hashing of the frames; any pair of frames within 3 bits is guaranteed to share a band, which is
    enough because near-identical videos have many such frames.
    Returns (a, b, match_fraction)."""
    usable = [i for i in ids if sigs.get(i) and sigs[i]["dhash"]]
    hashes = {i: np.array([int(h, 16) for h in sigs[i]["dhash"]], dtype=np.uint64) for i in usable}
    buckets: dict[tuple[int, int], set[int]] = defaultdict(set)
    for idx, vid in enumerate(usable):
        for h in hashes[vid].tolist():
            for band in range(4):
                buckets[(band, (h >> (16 * band)) & 0xFFFF)].add(idx)
    cand: Counter = Counter()
    for members in buckets.values():
        if 1 < len(members) <= 200:            # skip degenerate buckets (e.g. all-black frames)
            ml = sorted(members)
            for x in range(len(ml)):
                for y in range(x + 1, len(ml)):
                    cand[(ml[x], ml[y])] += 1
    pairs = []
    for (x, y), hits in cand.items():
        if hits < 2:
            continue
        A, B = hashes[usable[x]], hashes[usable[y]]
        ham = popcount64(A[:, None] ^ B[None, :])
        frac = max(float((ham.min(1) <= frame_bits).mean()), float((ham.min(0) <= frame_bits).mean()))
        if frac >= min_match_fraction and min(len(A), len(B)) >= 3:
            pairs.append((usable[x], usable[y], frac))
    return pairs


class UnionFind:
    def __init__(self):
        self.parent: dict[str, str] = {}

    def find(self, x: str) -> str:
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: str, b: str) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)


# --------------------------------------------------------------------------------------------
def place_file(src: Path, dst: Path, mode: str) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        if dst.stat().st_size == src.stat().st_size:
            return
        dst.unlink()
    if mode == "hardlink":
        try:
            os.link(src, dst)
            return
        except OSError:
            pass  # different filesystem: fall back to a copy
    if mode == "symlink":
        dst.symlink_to(src.resolve())
        return
    shutil.copy2(src, dst)


def quality_flags(rec: dict, sig: dict | None, min_short_side: int, min_seconds: float) -> list[str]:
    flags = []
    if min(rec["width"], rec["height"]) < min_short_side:
        flags.append("low_resolution")
    if rec["duration"] < min_seconds:
        flags.append("too_short")
    if rec["fps"] and not (14 <= rec["fps"] <= 61):
        flags.append("unusual_fps")
    if sig:
        if sig["motion"] < 2.0:
            flags.append("nearly_static")
        if sig["dark_fraction"] >= 0.5:
            flags.append("mostly_dark")
    else:
        flags.append("signature_failed")
    return flags


def split_of(group_id: str, val_fraction: float, seed: str) -> str:
    h = int(hashlib.sha256((seed + group_id).encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "val" if h < val_fraction else "train"


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", type=Path, default=Path("afterframe_data"), help="extract_afterframe.py --out folder")
    ap.add_argument("--out", type=Path, default=Path("dataset"))
    ap.add_argument("--permission-policy", choices=["owner-uploads", "allowlist", "all-attested", "pending"], required=True,
                    help="'pending' admits nothing: writes a status-only manifest while permissions are undecided")
    ap.add_argument("--uploader-id", action="append")
    ap.add_argument("--uploaders-file")
    ap.add_argument("--workers", type=int, default=max(1, os.cpu_count() or 2))
    ap.add_argument("--link-mode", choices=["hardlink", "copy", "symlink"], default="hardlink")
    ap.add_argument("--val-fraction", type=float, default=0.02)
    ap.add_argument("--split-seed", default="afterframe-v1")
    ap.add_argument("--near-dup-frame-bits", type=int, default=10,
                    help="dHash distance (0-64) at or below which two sampled frames count as the same")
    ap.add_argument("--near-dup-fraction", type=float, default=0.6,
                    help="share of one video's frames that must match the other to call them near-identical")
    ap.add_argument("--keep-near-duplicates", action="store_true",
                    help="keep every member of a near-duplicate group (still kept in the same split)")
    ap.add_argument("--group-identical-prompts", action=argparse.BooleanOptionalAction, default=True,
                    help="put videos with byte-identical prompts in one split group (prevents prompt leakage)")
    ap.add_argument("--min-short-side", type=int, default=360)
    ap.add_argument("--min-seconds", type=float, default=2.0)
    ap.add_argument("--exclude-flags", default="low_resolution,too_short,signature_failed",
                    help="comma list of quality flags that exclude a video; others are only recorded")
    ap.add_argument("--exclude-provenance", default="other_model_by_backend",
                    help="comma list of provenance labels to exclude (e.g. add provenance_uncertain)")
    args = ap.parse_args(argv)

    catalog, prompts = load_catalog(args.data), load_prompts(args.data)
    downloads = {r["id"]: r for r in read_jsonl(args.data / "state" / "downloads.jsonl")}
    policy = PermissionPolicy.from_args(args, None)
    excl_flags = {f for f in args.exclude_flags.split(",") if f}
    excl_prov = {f for f in args.exclude_provenance.split(",") if f}
    report: dict = {"generated_at": int(time.time() * 1000), "catalog_posts": len(catalog),
                    "prompt_lookups": len(prompts), "excluded": Counter(), "provenance": Counter(),
                    "quality_flags": Counter()}

    # 1. eligibility + presence of a verified download
    candidates: list[dict] = []
    for pid, post in catalog.items():
        ok, why = eligibility(post, prompts.get(pid), policy)
        if not ok:
            report["excluded"][why] += 1
            continue
        dl = downloads.get(pid)
        if not dl or dl.get("status") != "ok" or not Path(dl["path"]).exists():
            report["excluded"]["not_downloaded" if not dl else "download_" + str(dl.get("status"))] += 1
            continue
        candidates.append({"post": post, "prompt": prompts[pid], "dl": dl})
    print(f"{len(candidates)} eligible, downloaded videos; exclusions so far: {dict(report['excluded'])}")

    # 2. signatures
    sig_cache_path = args.out / ".signatures.json"
    sig_cache = json.loads(sig_cache_path.read_text()) if sig_cache_path.exists() else {}
    args.out.mkdir(parents=True, exist_ok=True)
    todo = {c["dl"]["sha256"]: c for c in candidates if c["dl"]["sha256"] not in sig_cache}
    with ThreadPoolExecutor(max_workers=args.workers) as pool:   # ffmpeg runs out of process
        futs = {pool.submit(frame_signature, Path(c["dl"]["path"]), c["dl"]["probe"]["duration"]): k
                for k, c in todo.items()}
        for n, fut in enumerate(as_completed(futs), 1):
            sig_cache[futs[fut]] = fut.result()
            if n % 200 == 0 or n == len(futs):
                print(f"  signatures {n}/{len(futs)}")
                sig_cache_path.write_text(json.dumps(sig_cache))
    sigs: dict[str, dict | None] = {c["post"]["id"]: sig_cache.get(c["dl"]["sha256"]) for c in candidates}

    # 3. duplicate groups (exact sha, same Sora generation id, near-identical frames, same prompt)
    uf = UnionFind()
    by_sha, by_gen, by_prompt = defaultdict(list), defaultdict(list), defaultdict(list)
    for c in candidates:
        pid = c["post"]["id"]
        uf.find(pid)
        by_sha[c["dl"]["sha256"]].append(pid)
        gen = sora_generation_id(c["post"]["data"].get("name"))
        if gen:
            by_gen[gen].append(pid)
        by_prompt[c["prompt"]["prompt"]].append(pid)
    hard_dup = UnionFind()  # groups whose members are the same video (deduplicated)
    for members in list(by_sha.values()) + list(by_gen.values()):
        for m in members[1:]:
            uf.union(members[0], m)
            hard_dup.union(members[0], m)
    near = near_duplicate_pairs([c["post"]["id"] for c in candidates], sigs, args.near_dup_frame_bits,
                                args.near_dup_fraction)
    for a, b, _ in near:
        uf.union(a, b)
        if not args.keep_near_duplicates:
            hard_dup.union(a, b)
    if args.group_identical_prompts:
        for members in by_prompt.values():
            for m in members[1:]:
                uf.union(members[0], m)
    report["near_duplicate_pairs"] = len(near)

    # keep one representative per hard-duplicate group: highest resolution, then has C2PA, then oldest
    def rank(c):
        p = c["dl"]["probe"]
        return (-(p["width"] * p["height"]), not c["dl"].get("c2pa", {}).get("present"), -p["duration"],
                c["post"]["data"].get("created", 0))
    reps: dict[str, dict] = {}
    for c in sorted(candidates, key=rank):
        root = hard_dup.find(c["post"]["id"])
        if root in reps:
            report["excluded"]["duplicate"] += 1
            continue
        reps[root] = c

    # 4. records
    rows = []
    for c in reps.values():
        post, data, dl, pr = c["post"], c["post"]["data"], c["dl"], c["prompt"]
        pid, probe = post["id"], dl["probe"]
        prov = provenance_assessment(data, probe, dl.get("c2pa"), pr)
        if prov["label"] in excl_prov:
            report["excluded"]["provenance_" + prov["label"]] += 1
            continue
        rec = {"width": probe["width"], "height": probe["height"], "fps": probe["fps"],
               "duration": probe["duration"]}
        flags = quality_flags(rec, sigs.get(pid), args.min_short_side, args.min_seconds)
        if prov["label"] != "sora2_likely" and prov["label"] != "sora2_confirmed_by_backend":
            flags.append("provenance_" + prov["label"])
        bad = sorted(set(flags) & excl_flags)
        if bad:
            report["excluded"]["quality_" + "+".join(bad)] += 1
            continue
        for f in flags:
            report["quality_flags"][f] += 1
        report["provenance"][prov["label"]] += 1
        shard = Path(dl["path"]).parent.name  # shard_XXX, as assigned at download time
        group = uf.find(pid)
        caption = make_caption(pr["prompt"])
        ext = Path(dl["path"]).suffix
        rows.append({
            "id": pid,
            "shard": shard,
            "video_path": f"{shard}/videos/{pid}{ext}",
            "caption_path": f"{shard}/captions/{pid}.txt",
            "backend_prompt": pr["prompt"],
            "caption_prefix": CAPTION_PREFIX,
            "training_caption": caption,
            "width": probe["width"], "height": probe["height"],
            "aspect_ratio": round(probe["width"] / probe["height"], 4),
            "fps": probe["fps"], "duration": round(probe["duration"], 4), "num_frames": probe["nb_frames"],
            "has_audio": probe["has_audio"], "video_codec": probe["vcodec"], "pix_fmt": probe["pix_fmt"],
            "bytes": dl["bytes"], "sha256": dl["sha256"],
            "sora_generation_id": sora_generation_id(data.get("name")),
            "provenance": prov,
            "quality_flags": sorted(flags),
            "motion_score": (sigs.get(pid) or {}).get("motion"),
            "dup_group": group,
            "split": split_of(group, args.val_fraction, args.split_seed),
            "source": {
                "site": BASE_URL, "post_id": pid, "asset_id": data.get("asset"),
                "media_url": f"{BASE_URL}/media/{data.get('asset')}",
                "original_filename": data.get("name"), "uploader_id": data.get("by"),
                "uploaded_at_ms": data.get("created"), "catalog_fetched_at_ms": post.get("fetched_at"),
                "prompt_fetched_at_ms": pr.get("fetched_at"), "downloaded_at_ms": dl.get("downloaded_at"),
                "permission_policy": policy.mode, "c2pa": dl.get("c2pa"),
            },
        })

    # 5. write files
    rows.sort(key=lambda r: (r["shard"], r["id"]))
    per_shard = defaultdict(list)
    for r in rows:
        src = Path(downloads[r["id"]]["path"])
        place_file(src, args.out / r["video_path"], args.link_mode)
        cap = args.out / r["caption_path"]
        cap.parent.mkdir(parents=True, exist_ok=True)
        cap.write_bytes(r["training_caption"].encode("utf-8"))
        per_shard[r["shard"]].append(r)
    header = {"_type": "manifest_header", "manifest_version": MANIFEST_VERSION, "caption_prefix": CAPTION_PREFIX,
              "records": len(rows), "generated_at_ms": report["generated_at"], "source": BASE_URL,
              "permission_policy": policy.mode, "catalog_posts": len(catalog), "prompt_lookups": len(prompts),
              "excluded": dict(report["excluded"]),
              "prompt_access": (json.loads((args.data / "state" / "prompt_access.json").read_text())
                                if (args.data / "state" / "prompt_access.json").exists() else None),
              "note": ("Records follow; each one has an authentic backend prompt. Lines with _type are metadata." if rows else
                       "No eligible records: see 'excluded' for the reason each catalog post was left out.")}
    with (args.out / "dataset_manifest.jsonl").open("w", encoding="utf-8") as fh:
        fh.write(json.dumps(header, ensure_ascii=False) + "\n")
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    for shard, srows in per_shard.items():
        with (args.out / shard / "manifest.jsonl").open("w", encoding="utf-8") as fh:
            for r in srows:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    (args.out / "splits").mkdir(exist_ok=True)
    for split in ("train", "val"):
        (args.out / "splits" / f"{split}.txt").write_text("".join(r["id"] + "\n" for r in rows if r["split"] == split))

    # leakage check: no group may appear in both splits
    group_splits = defaultdict(set)
    for r in rows:
        group_splits[r["dup_group"]].add(r["split"])
    leaks = [g for g, s in group_splits.items() if len(s) > 1]
    assert not leaks, f"split leakage in groups {leaks[:5]}"

    report.update({"records": len(rows), "train": sum(r["split"] == "train" for r in rows),
                   "val": sum(r["split"] == "val" for r in rows), "groups": len(group_splits),
                   "shards": {s: len(v) for s, v in sorted(per_shard.items())},
                   "total_hours": round(sum(r["duration"] for r in rows) / 3600, 3),
                   "total_gb": round(sum(r["bytes"] for r in rows) / 1e9, 3)})
    for k in ("excluded", "provenance", "quality_flags"):
        report[k] = dict(report[k])
    (args.out / "report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps({k: report[k] for k in ("records", "train", "val", "excluded", "provenance", "total_gb")}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
import importlib, json, sys, subprocess, glob, shutil
missing = [f for f in ("afterframe_wan.py", "extract_afterframe.py", "prepare_dataset.py")
           if not os.path.exists(f"{WORK}/{f}")]
assert not missing, (f"{missing} not found in {WORK}: run the three %%writefile cells above first "
                     "(they write to /kaggle/working; keep WORK = '/kaggle/working')")
sys.path.insert(0, WORK)
import afterframe_wan as aw
importlib.reload(aw)

def run(cmd, check=True):
    """Run a command, streaming its output into the notebook."""
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
    return p.returncode

def wan(*args, gpus=None):
    gpus = NUM_GPUS if gpus is None else gpus
    base = [sys.executable, f"{WORK}/afterframe_wan.py"] if gpus <= 1 else \
           ["torchrun", "--standalone", f"--nproc_per_node={gpus}", f"{WORK}/afterframe_wan.py"]
    return run(base + list(args))

def remaining_minutes():
    return (SESSION_HOURS * 3600 - (time.time() - NOTEBOOK_START)) / 60 - SAFETY_MINUTES

if HF_REPO:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print(json.dumps({k: {kk: v[kk] for kk in ("model", "rank", "alpha", "targets", "lr", "accum")} | {"stages": [s["name"] for s in v["stages"]]}
                  for k, v in aw.PRESETS.items()}, indent=1))

## 4 · (Optional) Extract Afterframe shards inside Kaggle

Skip this if the shards are already attached as Kaggle datasets (the output of `prepare_dataset.py`, one
`shard_XXX/` folder each). Otherwise this downloads a few shards at a time, so the whole archive never has
to fit in Kaggle's temporary storage. Prompts are owner-only on Afterframe: put the owner's signed-in
`Cookie` header in the Kaggle Secret `AFTERFRAME_COOKIE`. Choose the permission policy deliberately.

In [ ]:
EXTRACT_IN_KAGGLE = False
EXTRACT_SHARDS = "0,1"            # of 16 hash shards; each is ~1/16 of the archive
PERMISSION_ARGS = ["--permission-policy", "owner-uploads"]   # or: allowlist --uploaders-file ..., all-attested
if EXTRACT_IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["AFTERFRAME_COOKIE"] = UserSecretsClient().get_secret("AFTERFRAME_COOKIE")
    data = "/tmp/afterframe_data"
    run([sys.executable, f"{WORK}/extract_afterframe.py", "scan", "--out", data])
    run([sys.executable, f"{WORK}/extract_afterframe.py", "prompts", "--out", data])
    run([sys.executable, f"{WORK}/extract_afterframe.py", "download", "--out", data, "--shards", EXTRACT_SHARDS,
         "--num-shards", "16", "--deep-verify", *PERMISSION_ARGS])
    run([sys.executable, f"{WORK}/prepare_dataset.py", "--data", data, "--out", "/tmp/afterframe_dataset", *PERMISSION_ARGS])
    DATASET_ROOTS = ["/tmp/afterframe_dataset"] + DATASET_ROOTS

## 5 · Dataset discovery and plan

In [ ]:
shards = aw.discover_shards(DATASET_ROOTS)
assert shards, f"no shard_*/manifest.jsonl under {DATASET_ROOTS}"
cfg = aw.PRESETS[PRESET]; model_info = aw.MODELS[cfg["model"]]
fps = model_info["fps"]
summary, n_train_clips = {}, {i: 0 for i in range(len(cfg["stages"]))}
for name, path in sorted(shards.items()):
    rows = aw.read_manifest(path)           # also verifies every caption == prefix + backend prompt
    tr = [r for r in rows if r["split"] == "train"]
    summary[name] = {"videos": len(rows), "train": len(tr), "val": len(rows) - len(tr),
                     "hours": round(sum(r["duration"] for r in rows) / 3600, 2)}
    for i, st in enumerate(cfg["stages"]):
        n_train_clips[i] += sum(len(aw.plan_clips(r["duration"], st["frames"], fps, cfg["clips_per_video"])) for r in tr)
print(json.dumps(summary, indent=1))
ex = aw.read_manifest(next(iter(shards.values())))[0]
print("example caption:", ex["_caption"][:160])
assert ex["_caption"].startswith(aw.CAPTION_PREFIX)

fractions = STAGE_SHARD_FRACTION or [st.get("shard_fraction", 1.0) for st in cfg["stages"]]
world = max(1, NUM_GPUS)
steps_per_stage = [int(n_train_clips[i] * fractions[i] // (world * cfg["batch"] * cfg["accum"])) for i in n_train_clips]
TOTAL_STEPS = TOTAL_STEPS or max(100, sum(steps_per_stage))
print("train clips per stage:", n_train_clips, " optimizer steps per stage:", steps_per_stage, " cosine horizon:", TOTAL_STEPS)

## 6 · Base model download

In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download(model_info["repo"], allow_patterns=["model_index.json", "tokenizer/*", "text_encoder/*",
                                                                  "vae/*", "transformer/*", "scheduler/*"])
!du -sh {MODEL_DIR}/* | sort -h

## 7 · Smoke test (real model, a few clips, both GPUs)

This covers caching (with the VAE fp16-versus-fp32 check and the umT5 fp16 finiteness check), then 4 DDP
optimizer steps with a checkpoint, a resume to step 6, a validation video, the LoRA export, and loading
that export for inference. It runs in a separate cache and output folder, on the base model of the chosen
`PRESET`. With `PRESET = "wan22_5b_experimental"` this is the go/no-go test for the 5B model on T4: check the
printed peak memory and that no step reports a non-finite loss.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_cache, smoke_out = "/tmp/afterframe_smoke_cache", f"{WORK}/smoke_run"
    shutil.rmtree(smoke_out, ignore_errors=True)
    first = sorted(shards)[:1]
    # the smoke preset, run on the base model of the chosen PRESET (so wan22_5b_experimental is smoke-tested too)
    common = ["--preset", "smoke", "--model", cfg["model"], "--model-path", MODEL_DIR, "--cache-dir", smoke_cache]
    wan("cache", *common, "--dataset", *DATASET_ROOTS, "--shards", ",".join(first), "--max-videos-per-shard", "8")
    train = ["train", *common, "--output-dir", smoke_out, "--shards", ",".join(first), "--save-every", "4",
             "--log-every", "1", "--val-loss-every", "2", "--val-every", "4", "--val-steps", "8", "--passes", "100",
             "--total-steps", "6", "--warmup-steps", "1"]
    wan(*train, "--max-steps", "4")
    wan(*train, "--max-steps", "6")                       # must print "resumed from ... step_0000004"
    log = [json.loads(l) for l in open(f"{smoke_out}/logs/train_log.jsonl")]
    assert log[-1]["step"] == 6 and all(map(lambda r: r["loss"] == r["loss"], log)), "NaN loss in smoke test"
    print("peak GPU memory per rank (GB):", log[-1]["peak_mem_gb"])
    g = aw.Generator(cfg["model"], MODEL_DIR, lora=f"{smoke_out}/afterframe_sora2_lora.safetensors")
    v = g(aw.DEFAULT_VAL_PROMPTS[0], width=320, height=192, num_frames=17, steps=8, seed=0)
    aw.write_mp4(v, f"{smoke_out}/smoke_infer.mp4", model_info["fps"])
    del g; import gc; gc.collect(); torch.cuda.empty_cache()
    print("SMOKE TEST PASSED")

## 8 · Benchmark on these GPUs

The cell below measures the chosen preset on this hardware: latent caching time per clip, optimizer-step
time and peak memory, using the first shard and stage 0. It then estimates the GPU hours the whole plan
needs. All the figures come from this run.

In [ ]:
def bench_plan():
    bdir = f"{WORK}/bench_run"
    shutil.rmtree(bdir, ignore_errors=True)
    first = sorted(shards)[:1]
    est = {}
    for i, st in enumerate(cfg["stages"]):
        wan("cache", "--preset", PRESET, "--stage", str(i), "--model-path", MODEL_DIR, "--cache-dir", CACHE_DIR,
            "--dataset", *DATASET_ROOTS, "--shards", ",".join(first), "--max-videos-per-shard", str(4 * world))
        wan("train", "--preset", PRESET, "--stage", str(i), "--model-path", MODEL_DIR, "--cache-dir", CACHE_DIR,
            "--output-dir", f"{bdir}/s{i}", "--shards", ",".join(first), "--no-resume", "--max-steps", "6",
            "--log-every", "1", "--val-every", "0", "--val-loss-every", "0", "--save-every", "100000", "--passes", "100")
        log = [json.loads(l) for l in open(f"{bdir}/s{i}/logs/train_log.jsonl")]
        sec = sorted(r["sec_per_opt_step"] for r in log[2:])[len(log[2:]) // 2]   # median, after warm-up
        cache_b = [json.loads(l) for l in open(f"{CACHE_DIR}/benchmarks.jsonl")][-1]
        spc = cache_b["sec_per_clip"]
        est[st["name"]] = {"sec_per_opt_step": round(sec, 2), "peak_mem_gb": log[-1]["peak_mem_gb"],
                           "cache_sec_per_clip_per_gpu": [round(x, 2) for x in spc],
                           "planned_steps": steps_per_stage[i],
                           "train_gpu_hours_est": round(steps_per_stage[i] * sec * world / 3600, 1),
                           "cache_gpu_hours_est": round(n_train_clips[i] * fractions[i] * (sum(spc) / max(1, len(spc))) / 3600, 1)}
        shutil.rmtree(f"{CACHE_DIR}/{st['name']}", ignore_errors=True)   # benchmark clips are re-cached properly later
    json.dump(est, open(f"{OUTPUT_DIR}/benchmark_estimates.json", "w"), indent=1)
    return est

if RUN_BENCHMARK:
    est = bench_plan()
    print(json.dumps(est, indent=1))
    tot = sum(v["train_gpu_hours_est"] + v["cache_gpu_hours_est"] for v in est.values())
    print(f"whole plan: ~{tot:.0f} GPU-hours = ~{tot / world:.0f} wall-clock hours on {world} GPUs "
          f"(~{tot / world / SESSION_HOURS:.1f} Kaggle sessions)")

## 9 · Resume state from earlier sessions

To continue a run, attach the previous notebook version's output as an input dataset. This cell copies its
`afterframe_run/` back into `/kaggle/working` when it is further along. If `HF_REPO` is set, checkpoints are
also mirrored there during training.

In [ ]:
def run_step(path):
    try:
        return json.load(open(f"{path}/plan_state.json"))["global_step"]
    except Exception:
        return -1

candidates = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/afterframe_run/plan_state.json", recursive=True)]
best = max(candidates, key=run_step, default=None)
if best and run_step(best) > run_step(OUTPUT_DIR):
    print("restoring", best, "step", run_step(best))
    shutil.copytree(best, OUTPUT_DIR, dirs_exist_ok=True)
plan_path = f"{OUTPUT_DIR}/plan_state.json"
plan = json.load(open(plan_path)) if os.path.exists(plan_path) else {"preset": PRESET, "trained": {}, "global_step": 0}
assert plan["preset"] == PRESET, f"this run was started with preset {plan['preset']}"
print(json.dumps(plan, indent=1)[:1500])

## 10 · Training: progressive stages × shard chunks, time-budgeted

For each stage, and each chunk of `SHARDS_PER_CHUNK` shards not yet trained in that stage, the loop:
1. caches the chunk's latents and embeddings on both GPUs;
2. trains on it under DDP until the chunk is done or the session budget runs out;
3. records progress in `plan_state.json` and frees the chunk's cache.

When the session ends mid-chunk, the next session resumes at the same batch. Run *Save Version* (or keep
`HF_REPO` set) so `/kaggle/working/afterframe_run` survives.

In [ ]:
gpu_log = open(f"{OUTPUT_DIR}/gpu_monitor.csv", "a")
monitor = subprocess.Popen(["nvidia-smi", "--query-gpu=timestamp,index,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw",
                            "--format=csv,noheader", "-l", "30"], stdout=gpu_log)
names = sorted(shards)
chunks = [names[i:i + SHARDS_PER_CHUNK] for i in range(0, len(names), SHARDS_PER_CHUNK)]
extra = ["--hub-repo", HF_REPO] if HF_REPO else []
try:
    for si, st in enumerate(cfg["stages"]):
        stage_chunks = chunks[: max(1, round(len(chunks) * fractions[si]))]
        done = set(plan["trained"].get(st["name"], []))
        for chunk in stage_chunks:
            key = ",".join(chunk)
            if key in done:
                continue
            if remaining_minutes() < 30:
                raise StopIteration
            wan("cache", "--preset", PRESET, "--stage", str(si), "--model-path", MODEL_DIR, "--cache-dir", CACHE_DIR,
                "--dataset", *DATASET_ROOTS, "--shards", key)
            wan("train", "--preset", PRESET, "--stage", str(si), "--model-path", MODEL_DIR, "--cache-dir", CACHE_DIR,
                "--output-dir", OUTPUT_DIR, "--shards", key, "--total-steps", str(TOTAL_STEPS),
                "--max-train-minutes", str(int(remaining_minutes())), *extra)
            stop = json.load(open(f"{OUTPUT_DIR}/last_stop.json"))
            plan["global_step"] = stop["step"]
            if stop["reason"] == "chunk_done":
                plan["trained"].setdefault(st["name"], []).append(key)
                if DELETE_CACHE_AFTER_TRAINING:
                    for s in chunk:
                        shutil.rmtree(f"{CACHE_DIR}/{st['name']}/{s}", ignore_errors=True)
            json.dump(plan, open(plan_path, "w"), indent=1)
            if stop["reason"] != "chunk_done":
                raise StopIteration
    print("ALL STAGES AND SHARDS TRAINED")
except StopIteration:
    print(f"session budget reached at step {plan['global_step']}; save this version and continue next session")
finally:
    monitor.terminate(); gpu_log.close()
    json.dump(plan, open(plan_path, "w"), indent=1)

## 11 · Loss, validation loss and GPU memory

In [ ]:
import matplotlib.pyplot as plt, numpy as np, csv
def jl(p): return [json.loads(l) for l in open(p)] if os.path.exists(p) else []
tl, vl = jl(f"{OUTPUT_DIR}/logs/train_log.jsonl"), jl(f"{OUTPUT_DIR}/logs/val_log.jsonl")
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
if tl:
    s, l = np.array([r["step"] for r in tl]), np.array([r["loss"] for r in tl])
    ax[0].plot(s, l, alpha=.3, label="train (per log)")
    k = max(1, len(l) // 50); ax[0].plot(s[k - 1:], np.convolve(l, np.ones(k) / k, "valid"), label=f"train (avg {k})")
    for st in sorted({r["stage"] for r in tl}):
        first = next(r["step"] for r in tl if r["stage"] == st); ax[0].axvline(first, ls=":", c="gray")
    ax[1].plot(s, [r["lr"] for r in tl]); ax[1].set_title("learning rate")
    mem = np.array([r["peak_mem_gb"] for r in tl])
    for g in range(mem.shape[1]): ax[2].plot(s, mem[:, g], label=f"rank {g} peak allocated")
if vl:
    ax[0].plot([r["step"] for r in vl], [r["val_loss"] for r in vl], "o-", label="held-out (fixed noise)")
ax[0].set_title("flow-matching loss"); ax[0].legend(); ax[2].set_title("GPU memory (GB)")
if os.path.exists(f"{OUTPUT_DIR}/gpu_monitor.csv"):
    per_gpu = {}
    for r in csv.reader(open(f"{OUTPUT_DIR}/gpu_monitor.csv")):
        try:
            per_gpu.setdefault(r[1].strip(), []).append(float(r[3].split()[0]) / 1024)   # "1234 MiB" -> GB
        except (IndexError, ValueError):
            continue
    for g, used in sorted(per_gpu.items()):
        ax[2].plot(np.linspace(s.min() if tl else 0, s.max() if tl else 1, len(used)), used, ls="--", label=f"nvidia-smi GPU {g} (all processes)")
ax[2].axhline(15.0, c="r", ls=":"); ax[2].legend()
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=110); plt.show()
if tl:
    print({k: tl[-1][k] for k in ("step", "loss", "samples_per_s", "sec_per_opt_step", "peak_mem_gb", "nonfinite")})

## 12 · Validation videos written during training

In [ ]:
from IPython.display import HTML, display
import base64
vids = sorted(glob.glob(f"{OUTPUT_DIR}/validation/step_*/prompt*.mp4"))
for p in vids[-6:]:
    txt = open(p.replace(".mp4", ".txt")).read()
    b64 = base64.b64encode(open(p, "rb").read()).decode()
    display(HTML(f"<p><b>{p.split('/')[-2]}</b> — {txt}</p><video src='data:video/mp4;base64,{b64}' controls loop width=480></video>"))

## 13 · Export the final LoRA (.safetensors)

In [ ]:
LORA_PATH = f"{OUTPUT_DIR}/afterframe_sora2_lora.safetensors"
run([sys.executable, f"{WORK}/afterframe_wan.py", "export", "--checkpoint-roots", f"{OUTPUT_DIR}/checkpoints", "--out", LORA_PATH])
!ls -la {OUTPUT_DIR}/*.safetensors
if HF_REPO:
    from huggingface_hub import HfApi
    for f in [LORA_PATH, LORA_PATH.replace(".safetensors", "_comfyui.safetensors")]:
        HfApi().upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=HF_REPO)

## 14 · Inference: unseen prompts, base vs. fine-tuned

The transformer and VAE go on `cuda:0` and umT5 on `cuda:1`. Each prompt is generated twice with the same
seed: once with the LoRA disabled (base Wan) and once at `LORA_STRENGTH`.

In [ ]:
PROMPTS = [
    "Sora 2 video, prompt: A realistic handheld recording of an abandoned shopping mall at night.",
    "Sora 2 video, prompt: A golden retriever shakes off water after jumping out of a lake at sunset, slow motion.",
]
LORA_STRENGTH = 1.0
SEED = 1234
WIDTH, HEIGHT = 832, 480       # Wan2.1-1.3B native; multiples of 16 (32 for Wan2.2-5B)
NUM_FRAMES = 49                # 4k+1; 81 = 5 s at 16 fps
GUIDANCE = 5.0
STEPS = 30
COMPARE_WITH_BASE = True

gen = aw.Generator(cfg["model"], MODEL_DIR, lora=LORA_PATH)
os.makedirs(f"{WORK}/inference", exist_ok=True)
for i, prompt in enumerate(PROMPTS):
    kw = dict(seed=SEED + i, width=WIDTH, height=HEIGHT, num_frames=NUM_FRAMES, steps=STEPS, guidance=GUIDANCE)
    stem = f"{WORK}/inference/{i:02d}"
    t0 = time.time(); aw.write_mp4(gen(prompt, lora_strength=LORA_STRENGTH, **kw), f"{stem}_lora.mp4", model_info["fps"])
    print(f"[{i}] LoRA {time.time() - t0:.0f}s", end="  ")
    if COMPARE_WITH_BASE:
        t0 = time.time(); aw.write_mp4(gen(prompt, lora_strength=0, **kw), f"{stem}_base.mp4", model_info["fps"])
        print(f"base {time.time() - t0:.0f}s", end="")
        aw.side_by_side(f"{stem}_base.mp4", f"{stem}_lora.mp4", f"{stem}_compare.mp4")
    print()
    show = f"{stem}_compare.mp4" if COMPARE_WITH_BASE else f"{stem}_lora.mp4"
    b64 = base64.b64encode(open(show, "rb").read()).decode()
    display(HTML(f"<p>{prompt}<br>left: base Wan · right: LoRA ×{LORA_STRENGTH} · seed {SEED + i}</p>"
                 f"<video src='data:video/mp4;base64,{b64}' controls loop width=960></video>"))

### LoRA strength sweep (same prompt and seed)

In [ ]:
for s in [0.0, 0.5, 1.0, 1.25]:
    aw.write_mp4(gen(PROMPTS[0], lora_strength=s, seed=SEED, width=WIDTH, height=HEIGHT, num_frames=33, steps=STEPS,
                     guidance=GUIDANCE), f"{WORK}/inference/sweep_{s:.2f}.mp4", model_info["fps"])
print(sorted(glob.glob(f"{WORK}/inference/sweep_*.mp4")))